# Temporal Triadic Macrostates → Community Evolution: a reproducible experimental pipeline

**Pipeline version 3.0.** If the first output line does not say `Pipeline version: 3.0`, this is an old copy of the notebook.

This notebook re-runs the full study behind the manuscript *“Temporal Triadic Macrostates as Predictive and Early-Warning Indicators of Community Evolution in Dynamic Networks”* **from raw SNAP data**. No number from the draft is reused: every value that goes into the paper comes from the tables written to `RESULTS_DIR`.

**Run on a CPU runtime** (`Runtime → Change runtime type → CPU`; nothing here uses a GPU/TPU) with `Runtime → Run all`. Expect roughly 1.5–3 h with all datasets and robustness sweeps; `RUN_ROBUSTNESS=False` and `INCLUDE_STACKEXCHANGE=False` give a fast first pass. `SYNTHETIC_SMOKE_TEST=True` checks the whole pipeline in a few minutes without downloads.

### Datasets (SNAP, fetched by URL; all carry event timestamps)

| Key | SNAP file | Domain | Stride |
|---|---|---|---|
| `email-Eu-core` | email-Eu-core-temporal.txt.gz | institutional e-mail | 14 d |
| `CollegeMsg` | CollegeMsg.txt.gz | online social messaging | 7 d |
| `bitcoin-otc` | soc-sign-bitcoinotc.csv.gz | trust ratings | 30 d |
| `bitcoin-alpha` | soc-sign-bitcoinalpha.csv.gz | trust ratings | 30 d |
| `mathoverflow` | sx-mathoverflow.txt.gz | Q&A interactions | 30 d |
| `reddit-hyperlinks` | soc-redditHyperlinks-body/title.tsv | subreddit hyperlinks | 30 d |
| `askubuntu` *(INCLUDE_STACKEXCHANGE)* | sx-askubuntu.txt.gz | Q&A interactions | 30 d |
| `superuser` *(INCLUDE_STACKEXCHANGE)* | sx-superuser.txt.gz | Q&A interactions | 30 d |
| `wiki-talk` *(optional, large)* | wiki-talk-temporal.txt.gz | collaborative talk pages | 30 d |

### Analysis plan for v3 (fixed before the v3 run)
v2 ran on six networks. Its yield table and results exposed the problems below. Every change here was decided **before** v3 was run and is applied identically to every dataset. Changes that respond to something seen in v2 are marked †, so the manuscript can disclose them.

| v2 problem | v3 protocol |
|---|---|
| email-Eu-core crashed: no labelled instance fell in the last 25 % of snapshots, and the yield rule never checked the test period† | The yield rule now also requires ≥ `MIN_TEST_INSTANCES` evaluation instances (≥ `MIN_TEST_PER_CLASS` per class) and ≥ `MIN_TRAIN_INSTANCES` first-fold training instances. A dataset that cannot meet it is reported, not modelled. The yield table records where in time the instances fall. |
| 3 of 5 datasets were chosen with an *overlapping* label window (k < m), so up to 75 % of the label window's events were already in the feature window† | **Primary analysis uses non-overlapping labels only** (k = m). Candidates are (1,1), (2,2), (3,3), (4,4), (6,6) strides. Overlapping windows appear only in the robustness sweep. |
| Training labels could use events from inside the test period | **Purged split**: a training row must have every event it touches (feature windows *and* label window t+k, which spans strides t+k … t+k+m−1) strictly before the evaluation cut: `t + k + m ≤ cut`. |
| A single 25 % hold-out gave 47–70 test instances and CIs of about ±0.12 | **Rolling-origin (expanding-window) evaluation**: `N_EVAL_FOLDS` cuts spread evenly over the second half of the timeline. Each fold trains on the purged past and predicts the next block, and predictions are pooled, which roughly doubles the evaluated instances. Hyper-parameters are tuned once per feature set on the first fold's training data and then held fixed. |
| Random-forest seed noise (≈ 0.05 macro-F1) was as large as most ablation deltas† | Every RF is a **seed ensemble** (`N_RF_SEEDS` seeds, averaged class probabilities). The seed-to-seed s.d. is reported. |
| McNemar on accuracy does not test the claim of interest (incremental macro-F1) | **Paired community-cluster bootstrap** of the macro-F1 difference (combined − structural, combined − size-only, macrostate − structural, combined − persistence) on the same pooled instances. Ablation deltas use the same paired bootstrap. |
| Five per-dataset analyses, each underpowered, no pooled estimate | **Random-effects meta-analysis** (DerSimonian–Laird τ², Hartung–Knapp–Sidik–Jonkman CI and p) across datasets for each mechanism's primary coefficient and for the incremental macro-F1, with I². Holm across the 4 mechanisms at the pooled level. |
| HMM picked L = 5, the top of the grid, in every dataset† | Grid widened to L = 2…8, and a flag records whether the chosen L sits at the grid boundary. |
| Robustness ran on one dataset only (the configured one crashed) | Robustness sweeps run on **every** modelled dataset (window incl. overlapping labels, Jaccard θ, label threshold, Louvain seed, Leiden/LPA, death-as-shrink). |
| Communities that dissolve at t+k were silently dropped from labelling (selection) | Primary: unchanged (dropped). Sensitivity: `DEATH_AS_SHRINK=True` in the sweep. |
| ANOVA/Tukey p-values for closure ignored repeated measures of a community | Kept as descriptives. Inference on closure uses the cluster-robust logit (primary) and the cluster bootstrap. |
| Few replications† | Two more timestamped Q&A networks (askubuntu, superuser), on by default. |

Unchanged from v2: lagged features only (≤ t), triad definitions, cluster-robust SEs by community, Holm within dataset, BH across all dataset × mechanism tests, causal forward-filtered HMM fitted on the pre-evaluation period, survival analysis on post-birth communities.

## 0. Configuration

In [ ]:
# ======================= USER CONFIGURATION =======================
SYNTHETIC_SMOKE_TEST = False   # True -> run end-to-end on two small synthetic networks (no downloads)
RUN_ROBUSTNESS       = True    # Jaccard / label-threshold / seed / algorithm sweeps (slowest part)
RUN_LSTM             = True    # RF vs LSTM comparison (needs torch; preinstalled in Colab)
INCLUDE_WIKI_TALK    = False   # adds the large wiki-talk-temporal network (~7.8M events)
INCLUDE_STACKEXCHANGE = True   # adds sx-askubuntu and sx-superuser (two more Q&A replications)
SAVE_TO_DRIVE        = True   # copy results to Google Drive at the end

CONFIG = dict(
    SEED=42,
    MIN_COMM_SIZE=4,          # smallest community in which internal triads exist
    JACCARD_THETA=0.15,       # community-tracking threshold
    LABEL_THRESHOLD=0.10,     # +/-10% size change -> Growing / Shrinking
    DEATH_AS_SHRINK=False,    # primary: a community that disappears at t+k is unlabelled (sensitivity: True)
    CD_ALGO='louvain',        # 'louvain' | 'leiden' | 'lpa'
    CD_SEED=42,
    # --- rolling-origin evaluation ---
    EVAL_START=0.50,          # first evaluation cut at 50% of the snapshots
    N_EVAL_FOLDS=4,           # cuts spread evenly over the second half; predictions pooled
    N_CV_FOLDS=5,             # grouped CV inside the first fold's training data (tuning only)
    N_RS_ITER=25,             # randomized-search iterations per feature set
    N_RF_SEEDS=5,             # seed ensemble for every random forest
    N_BOOT=1000,              # cluster-bootstrap replicates
    N_ABLATION_SEEDS=3,
    HMM_STATES=[2, 3, 4, 5, 6, 7, 8],
    HMM_RESTARTS=5,
    LSTM_SEEDS=5,
    ROBUSTNESS_DATASETS='all',   # 'all' or a list of dataset keys
    # --- window selection (outcome-blind; Section 10b) ---
    # (m, k): window width = m x stride, label compares t with t+k. Primary candidates are NON-overlapping (k = m),
    # tried in this order; the first that passes the yield rule is used.
    WINDOW_CANDIDATES=[(1, 1), (2, 2), (3, 3), (4, 4), (6, 6)],
    ROBUSTNESS_WINDOWS=[(1, 1), (2, 2), (3, 3), (4, 4), (6, 6), (2, 1), (3, 1), (4, 1)],   # k < m: overlapping, sweep only
    MIN_INSTANCES=200,        # yield rule
    MIN_PER_CLASS=25,
    MIN_COMMUNITIES=10,       # >= grouped-CV folds; cluster-robust SEs need several clusters
    MIN_TEST_INSTANCES=60,    # labelled instances in the evaluation period (t >= first cut)
    MIN_TEST_PER_CLASS=10,
    MIN_TRAIN_INSTANCES=60,   # purged training instances before the first cut
    JACCARD_SWEEP=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    LABEL_SWEEP=[0.05, 0.10, 0.15, 0.20],
    CD_SEED_SWEEP=[0, 1, 2, 3, 4],
    CD_ALGO_SWEEP=['louvain', 'leiden', 'lpa'],
)

DATASETS = {
    'email-Eu-core': dict(url='https://snap.stanford.edu/data/email-Eu-core-temporal.txt.gz',
                          sep='ws', cols=(0, 1, 2), stride_days=14, domain='institutional e-mail'),
    'CollegeMsg':    dict(url='https://snap.stanford.edu/data/CollegeMsg.txt.gz',
                          sep='ws', cols=(0, 1, 2), stride_days=7, domain='online social messaging'),
    'bitcoin-otc':   dict(url='https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz',
                          sep=',', cols=(0, 1, 3), stride_days=30, domain='trust network'),
    'bitcoin-alpha': dict(url='https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz',
                          sep=',', cols=(0, 1, 3), stride_days=30, domain='trust network'),
    'mathoverflow':  dict(url='https://snap.stanford.edu/data/sx-mathoverflow.txt.gz',
                          sep='ws', cols=(0, 1, 2), stride_days=30, domain='Q&A interactions'),
    # subreddit -> subreddit hyperlinks (body + title files); nodes (subreddits) persist for years
    'reddit-hyperlinks': dict(urls=['https://snap.stanford.edu/data/soc-redditHyperlinks-body.tsv',
                                    'https://snap.stanford.edu/data/soc-redditHyperlinks-title.tsv'],
                          sep='\t', cols=(0, 1, 3), ts_format='datetime', stride_days=30,
                          domain='online community hyperlinks'),
}
if INCLUDE_STACKEXCHANGE:
    DATASETS['askubuntu'] = dict(url='https://snap.stanford.edu/data/sx-askubuntu.txt.gz',
                                 sep='ws', cols=(0, 1, 2), stride_days=30, domain='Q&A interactions')
    DATASETS['superuser'] = dict(url='https://snap.stanford.edu/data/sx-superuser.txt.gz',
                                 sep='ws', cols=(0, 1, 2), stride_days=30, domain='Q&A interactions')
if INCLUDE_WIKI_TALK:
    DATASETS['wiki-talk'] = dict(url='https://snap.stanford.edu/data/wiki-talk-temporal.txt.gz',
                                 sep='ws', cols=(0, 1, 2), stride_days=30, domain='collaborative talk pages')

if SYNTHETIC_SMOKE_TEST:
    DATASETS = {
        'synthA': dict(synthetic=dict(n_nodes=260, n_comms=8, days=420, events_per_day=160, seed=1),
                       stride_days=14, domain='synthetic A'),
        'synthB': dict(synthetic=dict(n_nodes=200, n_comms=6, days=300, events_per_day=140, seed=2),
                       stride_days=10, domain='synthetic B'),
    }
    CONFIG.update(N_RS_ITER=4, N_BOOT=100, N_ABLATION_SEEDS=2, N_RF_SEEDS=2, HMM_RESTARTS=2, LSTM_SEEDS=2,
                  HMM_STATES=[2, 3], N_EVAL_FOLDS=2, ROBUSTNESS_DATASETS=['synthA'], JACCARD_SWEEP=[0.10, 0.20],
                  LABEL_SWEEP=[0.10, 0.20], CD_SEED_SWEEP=[0, 1], CD_ALGO_SWEEP=['louvain', 'lpa'],
                  WINDOW_CANDIDATES=[(1, 1), (2, 2)], ROBUSTNESS_WINDOWS=[(1, 1), (2, 1)],
                  MIN_INSTANCES=100, MIN_PER_CLASS=5, MIN_COMMUNITIES=3,
                  MIN_TEST_INSTANCES=20, MIN_TEST_PER_CLASS=2, MIN_TRAIN_INSTANCES=20)

import os
import sys
IN_COLAB = 'google.colab' in sys.modules
BASE_DIR    = '/content' if IN_COLAB else os.getcwd()
DATA_DIR    = os.path.join(BASE_DIR, 'snap_data')
RESULTS_DIR = os.path.join(BASE_DIR, 'results_macrostates')
PIPELINE_VERSION = '3.0 (purged rolling-origin evaluation, non-overlapping labels, seed ensembles, meta-analysis)'
import shutil
if os.path.isdir(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)          # never mix files from an earlier run (e.g. a smoke test) into this one
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print('Pipeline version:', PIPELINE_VERSION)
print('Datasets:', list(DATASETS)); print('Results ->', RESULTS_DIR, '(cleared)')

## 1. Environment

In [ ]:
import sys, subprocess, importlib.util
def _ensure(pkgs):
    missing = [pip for pip, mod in pkgs if importlib.util.find_spec(mod) is None]
    if missing:
        print('Installing:', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
_ensure([('hmmlearn', 'hmmlearn'), ('shap', 'shap'), ('lifelines', 'lifelines'),
         ('statsmodels', 'statsmodels'), ('python-igraph', 'igraph'), ('leidenalg', 'leidenalg')])

In [ ]:
import os, gzip, json, math, time, shutil, pickle, random, warnings, urllib.request
from collections import Counter, defaultdict
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import logsumexp
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.inspection import permutation_importance
import shap
from hmmlearn.hmm import GaussianHMM
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
try:
    import torch, torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
try:
    import igraph as ig, leidenalg as la
    HAS_LEIDEN = True
except ImportError:
    HAS_LEIDEN = False

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
SEED = CONFIG['SEED']; random.seed(SEED); np.random.seed(SEED)
CLASSES = ['Growing', 'Stable', 'Shrinking']
# Okabe-Ito colour-blind-safe colours, fixed per class (never cycled)
CLASS_COLORS = {'Growing': '#009E73', 'Stable': '#7f7f7f', 'Shrinking': '#D55E00'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 9})
print('python', sys.version.split()[0], '| networkx', nx.__version__, '| shap', shap.__version__,
      '| torch' if HAS_TORCH else '| (no torch)', torch.__version__ if HAS_TORCH else '',
      '| leiden' if HAS_LEIDEN else '| (no leiden)')

## 2. Data loading and snapshot construction
Events `(u, v, t)` are read from the SNAP file and self-loops are dropped. Snapshots are **sliding windows** of width $w$ moved by a stride $s$ (when $w=s$ they are the usual disjoint windows). Each snapshot keeps (i) an undirected graph weighted by interaction count (used for community detection and triads) and (ii) the set of directed pairs (used for reciprocity). Only complete windows are kept.

**Why sliding windows.** In high-churn networks (trust ratings, Q&A, short-lived messaging), most users are active for only a few weeks. With disjoint windows the communities in two consecutive windows share almost no nodes, so Jaccard tracking breaks nearly every chain and almost no community can be followed across t−1, t, t+1 (the first full run kept only 3–19 instances for four of the five SNAP networks). Overlapping windows let tracking follow communities through member turnover.

**Label step.** With width $w = m\cdot s$, the label compares community size at $t$ with $t+k$. The **primary analysis always takes $k=m$**: the label window is the first one that **shares no events** with window $t$. Overlapping labels ($k<m$, the change in active membership over the next stride) appear only in the robustness sweep. Features always use windows ≤ t only.

In [ ]:
def download(url):
    path = os.path.join(DATA_DIR, os.path.basename(url))
    if not os.path.exists(path):
        print('  downloading', url)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (research pipeline)'})
        with urllib.request.urlopen(req, timeout=300) as r, open(path + '.part', 'wb') as f:
            shutil.copyfileobj(r, f)
        os.replace(path + '.part', path)
    return path

def synthetic_events(n_nodes, n_comms, days, events_per_day, seed):
    # Planted communities whose membership drifts (nodes migrate, communities grow and shrink).
    rng = np.random.default_rng(seed)
    memb = rng.integers(0, n_comms, n_nodes)
    pull = rng.normal(0, 1, n_comms)
    rows = []
    for d in range(days):
        if d % 7 == 0:                        # weekly migration biased by a slowly drifting 'pull'
            pull = 0.9 * pull + rng.normal(0, 0.5, n_comms)
            p = np.exp(pull) / np.exp(pull).sum()
            movers = rng.random(n_nodes) < 0.04
            memb[movers] = rng.choice(n_comms, movers.sum(), p=p)
        for _ in range(events_per_day):
            u = rng.integers(n_nodes)
            same = np.flatnonzero(memb == memb[u])
            v = rng.choice(same) if (rng.random() < 0.85 and len(same) > 1) else rng.integers(n_nodes)
            if u != v:
                rows.append((u, v, d * 86400 + rng.integers(86400)))
                if rng.random() < 0.3: rows.append((v, u, d * 86400 + rng.integers(86400)))
    return pd.DataFrame(rows, columns=['src', 'dst', 'ts'])

def load_events(name, spec):
    if 'synthetic' in spec:
        ev = synthetic_events(**spec['synthetic'])
    else:
        sep = r'\s+' if spec['sep'] == 'ws' else spec['sep']
        frames = []
        for url in spec.get('urls', [spec.get('url')]):
            f = pd.read_csv(download(url), sep=sep, header=None, comment=None if spec.get('ts_format') else '#',
                            usecols=list(spec['cols']), dtype=str)
            f.columns = ['src', 'dst', 'ts']; frames.append(f)
        ev = pd.concat(frames, ignore_index=True)
        if spec.get('ts_format') == 'datetime':
            ts = pd.to_datetime(ev.ts, errors='coerce')            # header rows -> NaT, dropped below
            ev = ev.assign(ts=(ts - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')).dropna()
        else:
            ev = ev.assign(ts=pd.to_numeric(ev.ts, errors='coerce'))
        ev = ev.dropna()
        if not (pd.to_numeric(ev.src, errors='coerce').notna().all() and pd.to_numeric(ev.dst, errors='coerce').notna().all()):
            codes, _ = pd.factorize(pd.concat([ev.src, ev.dst], ignore_index=True))   # string node ids -> ints
            ev = ev.assign(src=codes[:len(ev)], dst=codes[len(ev):])
    ev = ev.dropna()
    ev = ev[ev.src != ev.dst].astype({'src': 'int64', 'dst': 'int64', 'ts': 'int64'})
    return ev.sort_values('ts').reset_index(drop=True)

def build_snapshots(ev, width_days, stride_days):
    # Window k covers [t0 + k*stride, t0 + k*stride + width). Only complete windows are kept.
    w, s = int(width_days * 86400), int(stride_days * 86400)
    t0, T = ev.ts.min(), ev.ts.max() - ev.ts.min()
    n_win = max(1, int((T - w) // s) + 1)
    ts = ev.ts.values; snaps = []
    for k in range(n_win):
        lo, hi = np.searchsorted(ts, [t0 + k * s, t0 + k * s + w], side='left')
        grp = ev.iloc[lo:hi]
        dcount = grp.groupby(['src', 'dst']).size()
        directed = set(dcount.index)
        und = Counter()
        for (u, v), c in dcount.items():
            und[(u, v) if u < v else (v, u)] += c
        G = nx.Graph(); G.add_weighted_edges_from((u, v, c) for (u, v), c in und.items())
        snaps.append(dict(G=G, directed=directed, n_events=len(grp)))
    return snaps

## 3. Community detection and tracking
Communities are detected per snapshot (Louvain by default; Leiden and asynchronous label propagation are available for robustness) and only communities with ≥ `MIN_COMM_SIZE` nodes are kept. Tracking matches communities in consecutive snapshots by **greedy one-to-one Jaccard matching** above θ; unmatched communities are births, unmatched predecessors are deaths.

In [ ]:
def detect_communities(G, algo='louvain', seed=42, min_size=4):
    if G.number_of_edges() == 0:
        return []
    if algo == 'louvain':
        comms = nx.community.louvain_communities(G, weight='weight', seed=seed)
    elif algo == 'leiden':
        if not HAS_LEIDEN: raise RuntimeError('leidenalg not installed')
        g = ig.Graph.from_networkx(G)
        part = la.find_partition(g, la.ModularityVertexPartition, weights='weight', seed=seed)
        names = g.vs['_nx_name']
        comms = [{names[i] for i in c} for c in part]
    elif algo == 'lpa':
        comms = nx.community.asyn_lpa_communities(G, weight='weight', seed=seed)
    else:
        raise ValueError(algo)
    return sorted([frozenset(c) for c in comms if len(c) >= min_size], key=lambda c: -len(c))

def track_communities(parts, theta):
    # Returns ids[t][j] = persistent community id of parts[t][j].
    ids, next_id, prev, prev_ids = [], 0, None, None
    for comms in parts:
        cur = [None] * len(comms)
        if prev:
            node2prev = {u: i for i, c in enumerate(prev) for u in c}
            cands = []
            for j, c in enumerate(comms):
                ov = Counter(node2prev[u] for u in c if u in node2prev)
                for i, o in ov.items():
                    jac = o / (len(prev[i]) + len(c) - o)
                    if jac >= theta: cands.append((jac, i, j))
            used = set()
            for jac, i, j in sorted(cands, reverse=True):
                if i in used or cur[j] is not None: continue
                cur[j] = prev_ids[i]; used.add(i)
        for j in range(len(comms)):
            if cur[j] is None: cur[j] = next_id; next_id += 1
        ids.append(cur); prev, prev_ids = comms, cur
    return ids

## 4. Community-conditioned triadic macrostates
For community $C$ in snapshot $G_t$ we count every **connected** node triple with at least two members in $C$, in five macrostates:

| State | Members in C | Configuration | Interpretation |
|---|---|---|---|
| S1 | 3 | closed (triangle) | internal cohesion |
| S2 | 3 | open (wedge) | internal, not yet closed |
| S3 | 2 | closed (triangle with one outsider) | boundary embedding |
| S4 | 2 | open, outsider is the centre (broker) | outsider bridging two members |
| S5 | 2 | open, member is the centre | member reaching out |

Counts are exact (no sampling). Derived quantities: shares $\tilde m^{(k)}$, Shannon entropy $H_t$, internal closure rate $=S1/(S1+S2)$, boundary ratio $=(S3+S4+S5)/\sum_k S_k$, lagged boundary closure rate $BCR_t=(C_b(t)-C_b(t-1))/O_b(t-1)$ with $C_b=S3$, $O_b=S4+S5$; and structural descriptors (size, density, conductance, reciprocity).

In [ ]:
MACRO_K = 5

def community_stats(G, directed, C):
    C = set(C); H = G.subgraph(C)
    n, m = len(C), H.number_of_edges()
    tri = sum(nx.triangles(H).values()) // 3
    wedges = sum(d * (d - 1) // 2 for _, d in H.degree())
    s1, s2 = tri, wedges - 3 * tri                       # node-triples: closed / open
    out = {u: {v for v in G[u] if v not in C} for u in C}
    kx = Counter(x for u in C for x in out[u])
    pairs_x = sum(k * (k - 1) // 2 for k in kx.values())
    s3 = s5 = 0
    for a, b in H.edges():
        c = len(out[a] & out[b]); s3 += c
        s5 += len(out[a]) + len(out[b]) - 2 * c
    s4 = pairs_x - s3
    counts = np.array([s1, s2, s3, s4, s5], dtype=float)
    tot = counts.sum()
    shares = counts / tot if tot > 0 else np.zeros(MACRO_K)
    nz = shares[shares > 0]
    cut = sum(len(o) for o in out.values())
    vol_c = sum(G.degree(u) for u in C); vol_r = 2 * G.number_of_edges() - vol_c
    recip = (sum(1 for a, b in H.edges() if (a, b) in directed and (b, a) in directed) / m) if m else 0.0
    return dict(size=n, density=2 * m / (n * (n - 1)), conductance=cut / max(1, min(vol_c, vol_r)),
                reciprocity=recip, counts=counts, shares=shares,
                entropy=float(-(nz * np.log(nz)).sum()),
                closure_rate=s1 / (s1 + s2) if (s1 + s2) > 0 else 0.0,
                boundary_ratio=(s3 + s4 + s5) / tot if tot > 0 else 0.0,
                Cb=s3, Ob=s4 + s5)

STRUCT = ['log_size', 'density', 'conductance', 'reciprocity', 'rel_d_size']
SHARES = [f'S{k+1}_share' for k in range(MACRO_K)]
MACRO  = SHARES + ['entropy', 'd_entropy', 'closure_rate', 'd_closure_rate',
                   'boundary_ratio', 'd_boundary_ratio', 'BCR']
COMBINED = STRUCT + MACRO
SIZE_ONLY = ['log_size', 'rel_d_size']
FEATURE_SETS = {'size only': SIZE_ONLY, 'structural': STRUCT, 'macrostate': MACRO, 'combined': COMBINED}
ABLATION_GROUPS = {
    'size (log_size, rel_d_size)': ['log_size', 'rel_d_size'],
    'density': ['density'],
    'conductance + reciprocity': ['conductance', 'reciprocity'],
    'internal closure (S1, S2, closure rate, d_closure)': ['S1_share', 'S2_share', 'closure_rate', 'd_closure_rate'],
    'boundary (S3-S5, ratio, d_ratio, BCR)': ['S3_share', 'S4_share', 'S5_share', 'boundary_ratio', 'd_boundary_ratio', 'BCR'],
    'entropy (H, dH)': ['entropy', 'd_entropy'],
    'ALL macrostate features': MACRO,
    'ALL structural features': STRUCT,
}

def snapshot_stats(snaps, parts):
    return [[community_stats(s['G'], s['directed'], c) for c in comms] for s, comms in zip(snaps, parts)]

def label_from_change(r, thr):
    return 'Growing' if r > thr else ('Shrinking' if r < -thr else 'Stable')

def build_table(stats, ids, n_snap, label_thr, death_as_shrink=False, h=1):
    # One row per (community id, snapshot). Features need t-1; the label compares t with t+h
    # (h = label step k; h = m means the label window shares no events with window t).
    rec = {(cid, t): s for t, (ss, ii) in enumerate(zip(stats, ids)) for s, cid in zip(ss, ii)}
    rows = []
    for (cid, t), s in rec.items():
        row = dict(cid=cid, t=t, size=s['size'], log_size=np.log(s['size']), density=s['density'],
                   conductance=s['conductance'], reciprocity=s['reciprocity'], entropy=s['entropy'],
                   closure_rate=s['closure_rate'], boundary_ratio=s['boundary_ratio'],
                   n_triads=s['counts'].sum(), Cb=s['Cb'], Ob=s['Ob'])
        row.update({k: v for k, v in zip(SHARES, s['shares'])})
        p = rec.get((cid, t - 1))
        if p is not None:
            row.update(rel_d_size=(s['size'] - p['size']) / p['size'],
                       d_entropy=s['entropy'] - p['entropy'],
                       d_closure_rate=s['closure_rate'] - p['closure_rate'],
                       d_boundary_ratio=s['boundary_ratio'] - p['boundary_ratio'],
                       BCR=(s['Cb'] - p['Cb']) / max(p['Ob'], 1.0))
        ph = rec.get((cid, t - h))
        if ph is not None: row['rel_d_size_h'] = (s['size'] - ph['size']) / ph['size']
        nx_ = rec.get((cid, t + h))
        n_next = nx_['size'] if nx_ is not None else (0 if (death_as_shrink and t + h < n_snap) else None)
        if n_next is not None:
            r = (n_next - s['size']) / s['size']
            row.update(rel_change_next=r, vol_next=abs(r), label=label_from_change(r, label_thr))
        row['born'] = (cid, t - 1) not in rec
        row['dies_next'] = (cid, t + 1) not in rec and t + 1 < n_snap
        rows.append(row)
    tab = pd.DataFrame(rows).sort_values(['cid', 't']).reset_index(drop=True)
    tab['age'] = tab.t - tab.groupby('cid').t.transform('min')   # snapshots since first tracked
    for c in ['rel_d_size', 'rel_d_size_h', 'd_entropy', 'd_closure_rate', 'd_boundary_ratio', 'BCR',
              'rel_change_next', 'vol_next', 'label']:
        if c not in tab: tab[c] = np.nan
    return tab

def instances_from(tab):
    return tab.dropna(subset=COMBINED + ['label']).reset_index(drop=True)

## 5. Evaluation protocol
* **Rolling origin with purging.** With $T$ snapshots, evaluation cuts $c_0<c_1<\dots$ are spread evenly from $\lfloor 0.5\,T\rfloor$ to $T$. Fold $i$ tests on instances with $c_i \le t < c_{i+1}$ and trains on instances whose every event lies before $c_i$: $t+k+m \le c_i$ (the label window $t+k$ spans strides $t+k,\dots,t+k+m-1$). Predictions from all folds are pooled.
* **Tuning:** randomized search over RF hyper-parameters, scored by macro-F1 under **GroupKFold by community id** on the first fold's training data. It runs once per feature set, and those parameters are reused in every fold.
* **Seed ensembles:** each RF averages class probabilities over `N_RF_SEEDS` seeds.
* **Uncertainty:** 95 % **cluster bootstrap** (resampling communities) for every metric. **Paired** cluster bootstrap for differences between models evaluated on the same instances.
* **Baselines:** majority class (per fold), stratified random (mean of 200 draws from each fold's training class mix), persistence (last observed size change, same ±threshold rule), multinomial logistic regression.

In [ ]:
RF_SPACE = dict(n_estimators=[200, 400, 600], max_depth=[None, 4, 6, 8, 12],
                min_samples_leaf=[1, 2, 4, 8], max_features=['sqrt', 0.5, None],
                class_weight=[None, 'balanced', 'balanced_subsample'])

def eval_cuts(n_snap):
    s, F = CONFIG['EVAL_START'], CONFIG['N_EVAL_FOLDS']
    cuts = sorted(set(int(np.floor(n_snap * (s + i * (1 - s) / F))) for i in range(F)))
    return cuts + [n_snap]

def purged_train(df, cut, m, k):
    # every event a training row touches (feature windows <= t, label window t+k spanning strides
    # t+k .. t+k+m-1) lies strictly before stride `cut`
    return df[df.t + k + m <= cut]

def make_folds(df, n_snap, m, k):
    c, out = eval_cuts(n_snap), []
    for i in range(len(c) - 1):
        tr = purged_train(df, c[i], m, k); te = df[(df.t >= c[i]) & (df.t < c[i + 1])]
        if len(te) and len(tr) and tr.label.nunique() >= 2 and tr.cid.nunique() >= 2:
            out.append((i, c[i], tr, te))
    return out

def temporal_split(df, n_snap, m, k):
    # single purged split at the first evaluation cut (used by transfer and RF-vs-LSTM)
    c0 = eval_cuts(n_snap)[0]
    return purged_train(df, c0, m, k), df[df.t >= c0], c0

def n_group_folds(groups, k):
    return max(2, min(k, len(np.unique(groups))))

def tune_rf(X, y, groups, seed=SEED):
    cv = GroupKFold(n_splits=n_group_folds(groups, CONFIG['N_CV_FOLDS']))
    rs = RandomizedSearchCV(RandomForestClassifier(random_state=seed, n_jobs=-1), RF_SPACE,
                            n_iter=CONFIG['N_RS_ITER'], scoring='f1_macro', cv=cv,
                            random_state=seed, n_jobs=1, refit=True)
    rs.fit(X, y, groups=groups)
    return rs.best_estimator_, rs.best_params_, rs.best_score_

def macro_f1(y, yhat):
    return f1_score(y, yhat, labels=CLASSES, average='macro', zero_division=0)

def metrics(y, yhat):
    return dict(accuracy=accuracy_score(y, yhat), macro_f1=macro_f1(y, yhat),
                balanced_acc=balanced_accuracy_score(y, yhat))

def _boot_index(groups, n_boot, seed):
    rng = np.random.default_rng(seed); groups = np.asarray(groups)
    ug = np.unique(groups); idx = {g: np.flatnonzero(groups == g) for g in ug}
    for _ in range(n_boot):
        yield np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)])

def cluster_bootstrap_ci(y, yhat, groups, n_boot=None, seed=SEED):
    y, yhat = np.asarray(y), np.asarray(yhat); out = defaultdict(list)
    for sel in _boot_index(groups, n_boot or CONFIG['N_BOOT'], seed):
        for k, v in metrics(y[sel], yhat[sel]).items(): out[k].append(v)
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in out.items()}

def paired_cluster_boot(y, pa, pb, groups, n_boot=None, seed=SEED):
    # macro-F1(a) - macro-F1(b) on the same instances; communities resampled; two-sided bootstrap p
    y, pa, pb = map(np.asarray, (y, pa, pb))
    obs = macro_f1(y, pa) - macro_f1(y, pb)
    d = np.array([macro_f1(y[s], pa[s]) - macro_f1(y[s], pb[s]) for s in _boot_index(groups, n_boot or CONFIG['N_BOOT'], seed)])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return obs, (np.percentile(d, 2.5), np.percentile(d, 97.5)), p

def rf_seed_ensemble(tr, te, cols, params, seeds):
    # average class probabilities over seeds; returns ensemble labels, per-seed labels, first fitted model
    P = np.zeros((len(te), len(CLASSES))); per_seed, first = [], None
    for s in seeds:
        est = RandomForestClassifier(random_state=s, n_jobs=-1, **params).fit(tr[cols], tr.label)
        first = est if first is None else first
        pr, full = est.predict_proba(te[cols]), np.zeros((len(te), len(CLASSES)))
        for j, c in enumerate(est.classes_): full[:, CLASSES.index(c)] = pr[:, j]
        P += full; per_seed.append(np.array(CLASSES)[full.argmax(1)])
    return np.array(CLASSES)[P.argmax(1)], per_seed, first

def persistence_baseline(df, thr):
    # last observed change over the same horizon as the label (t-h -> t); falls back to t-1 -> t
    r = df.rel_d_size_h.fillna(df.rel_d_size)
    return r.apply(lambda v: label_from_change(v, thr)).values

def fmt_ci(v, ci): return f'{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]'

## 6. Statistical helpers for the four mechanisms
Each mechanism has one **pre-specified primary test and expected sign**; the verdict is *supported* (Holm-adjusted p < 0.05, expected sign), *reversed* (Holm p < 0.05, opposite sign) or *not supported*.

| Mechanism | Primary test | Expected sign |
|---|---|---|
| M1 Boundary closure → growth | Logit(Growing vs rest) on z-scored Δboundary_ratio, BCR, boundary_ratio, log size, density, rel Δsize; cluster-robust SE | Δboundary_ratio **+** |
| M2 Internal closure → trajectory | Logit(Growing vs Shrinking) on z-scored closure rate + log size + density, cluster-robust (ANOVA / Kruskal–Wallis / Tukey / Cohen's d / cluster-bootstrap reported alongside) | closure **+** |
| M3 Entropy change → shrinkage | Logit(Shrinking vs rest) on z-scored H, ΔH, log size, rel Δsize; cluster-robust | ΔH **+** (diversification-instability prior) |
| M4 Regime persistence → stability | OLS log(1+\|Δn/n\|) on z-scored filtered regime run-length, expected regime duration, log size, **community age**; cluster-robust | run length **−** |

In [ ]:
def zscore(df, cols):
    sd = df[cols].std(ddof=0).replace(0, 1)
    return (df[cols] - df[cols].mean()) / sd

def cluster_fit(df, y, xs, kind='logit', group='cid'):
    d = df[[y] + xs + [group]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < len(xs) + 10 or d[y].nunique() < 2:
        return None, None
    X = sm.add_constant(zscore(d, xs)); g = pd.factorize(d[group])[0]
    model = sm.Logit(d[y].astype(float), X) if kind == 'logit' else sm.OLS(d[y].astype(float), X)
    try:
        res = model.fit(disp=0, maxiter=300, cov_type='cluster', cov_kwds={'groups': g}) if kind == 'logit' \
              else model.fit(cov_type='cluster', cov_kwds={'groups': g})
    except Exception as e:
        print('   fit failed:', e); return None, None
    ci = res.conf_int()
    tab = pd.DataFrame({'coef_perSD': res.params, 'se_cluster': res.bse, 'p': res.pvalues,
                        'ci_low': ci[0], 'ci_high': ci[1]})
    if kind == 'logit':
        tab['OR_perSD'] = np.exp(tab.coef_perSD); tab['OR_low'] = np.exp(tab.ci_low); tab['OR_high'] = np.exp(tab.ci_high)
        fit = dict(n=int(res.nobs), n_clusters=int(len(np.unique(g))), pseudo_R2=res.prsquared, llr_p=res.llr_pvalue)
    else:
        fit = dict(n=int(res.nobs), n_clusters=int(len(np.unique(g))), R2=res.rsquared)
    return tab.drop(index='const'), fit

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def cluster_boot_diff(df, col, g1, g2, n_boot=None, seed=SEED):
    n_boot = n_boot or CONFIG['N_BOOT']; rng = np.random.default_rng(seed)
    cids = df.cid.unique(); by = {c: d for c, d in df.groupby('cid')}
    diffs = []
    for _ in range(n_boot):
        s = pd.concat([by[c] for c in rng.choice(cids, len(cids), replace=True)])
        a, b = s.loc[s.label == g1, col], s.loc[s.label == g2, col]
        if len(a) and len(b): diffs.append(a.mean() - b.mean())
    return np.percentile(diffs, [2.5, 97.5])

def verdict(p_holm, sign, expected):
    if p_holm is None or np.isnan(p_holm): return 'not testable'
    if p_holm >= 0.05: return 'not supported'
    return 'supported' if np.sign(sign) == np.sign(expected) else 'reversed (significant, opposite sign)'

def meta_random_effects(est, se, se_floor=1e-3):
    # DerSimonian-Laird tau^2 with Hartung-Knapp-Sidik-Jonkman CI / p (t with k-1 df; HKSJ variance never
    # smaller than the DL variance). Appropriate for a small number of studies (here: datasets).
    # se_floor: a bootstrap SE of exactly 0 (two models with identical predictions) would get infinite weight.
    est, se = np.asarray(est, float), np.asarray(se, float)
    ok = np.isfinite(est) & np.isfinite(se); est, se = est[ok], np.maximum(se[ok], se_floor); k = len(est)
    if k < 2: return None
    w = 1 / se ** 2; fe = (w * est).sum() / w.sum(); Q = (w * (est - fe) ** 2).sum()
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    wr = 1 / (se ** 2 + tau2); mu = (wr * est).sum() / wr.sum()
    var_dl = 1 / wr.sum(); var_hk = (wr * (est - mu) ** 2).sum() / ((k - 1) * wr.sum())
    se_mu = np.sqrt(max(var_dl, var_hk)); tq = stats.t.ppf(0.975, k - 1)
    return dict(k=k, pooled=mu, se=se_mu, ci_low=mu - tq * se_mu, ci_high=mu + tq * se_mu,
                p=2 * stats.t.sf(abs(mu / se_mu), k - 1), tau2=tau2, I2=max(0.0, (Q - (k - 1)) / Q) if Q > 0 else 0.0, Q=Q)

## 7. Latent-regime (HMM) model
Gaussian HMM (diagonal covariance) on the z-scored macrostate shares S1–S4 (S5 is dropped because shares sum to 1). The number of states $L$ is chosen by BIC over `HMM_STATES`, each with several random restarts, **using only training-period snapshots**. Regimes are then read off with a **forward filter** (only data up to $t$), giving for every community-snapshot the filtered regime, its current run length, the regime's expected duration $1/(1-A_{ii})$ and the switching rate so far.

In [ ]:
HMM_COLS = SHARES[:-1]

def _sequences(tab, cols, t_max=None):
    seqs, keys = [], []
    for cid, d in tab.groupby('cid'):
        d = d.sort_values('t')
        if t_max is not None: d = d[d.t < t_max]
        if len(d) >= 2:
            seqs.append(d[cols].values); keys.append((cid, d.t.values))
    return seqs, keys

def fit_hmm(tab, cut):
    seqs, _ = _sequences(tab, HMM_COLS, t_max=cut)
    X = np.vstack(seqs); lengths = [len(s) for s in seqs]
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Xz = (X - mu) / sd; D = Xz.shape[1]
    rows, best = [], None
    for L in CONFIG['HMM_STATES']:
        if len(Xz) < 10 * L: continue
        bestL = None
        for r in range(CONFIG['HMM_RESTARTS']):
            m = GaussianHMM(n_components=L, covariance_type='diag', n_iter=300, tol=1e-4, random_state=SEED + r)
            try:
                m.fit(Xz, lengths); ll = m.score(Xz, lengths)
            except Exception:
                continue
            if bestL is None or ll > bestL[1]: bestL = (m, ll)
        if bestL is None: continue
        k = (L - 1) + L * (L - 1) + 2 * L * D
        bic = -2 * bestL[1] + k * np.log(len(Xz))
        rows.append(dict(L=L, logL=bestL[1], n_params=k, BIC=bic))
        if best is None or bic < best[2]: best = (bestL[0], L, bic)
    return best[0], pd.DataFrame(rows), (mu, sd)

def _log_emission(model, X):
    cov = np.asarray(model.covars_)
    var = np.array([np.diag(c) for c in cov]) if cov.ndim == 3 else cov
    return -0.5 * (np.log(2 * np.pi * var)[None] + (X[:, None, :] - model.means_[None]) ** 2 / var[None]).sum(-1)

def forward_filter(model, X):
    logB = _log_emission(model, X)
    logA = np.log(model.transmat_ + 1e-300); a = np.log(model.startprob_ + 1e-300) + logB[0]
    out = [a - logsumexp(a)]
    for t in range(1, len(X)):
        a = logsumexp(out[-1][:, None] + logA, axis=0) + logB[t]
        out.append(a - logsumexp(a))
    return np.exp(np.array(out))

def add_regime_features(tab, model, norm):
    mu, sd = norm
    exp_dur = 1.0 / (1.0 - np.clip(np.diag(model.transmat_), 0, 1 - 1e-6))
    rec = []
    for cid, d in tab.groupby('cid'):
        d = d.sort_values('t')
        post = forward_filter(model, (d[HMM_COLS].values - mu) / sd)
        st = post.argmax(1); run = np.ones(len(st)); sw = np.zeros(len(st))
        for i in range(1, len(st)):
            run[i] = run[i - 1] + 1 if st[i] == st[i - 1] else 1
            sw[i] = sw[i - 1] + (st[i] != st[i - 1])
        for i, t in enumerate(d.t.values):
            rec.append(dict(cid=cid, t=t, regime=int(st[i]), regime_run=run[i],
                            regime_exp_dur=exp_dur[st[i]],
                            switch_rate=sw[i] / i if i > 0 else np.nan,
                            **{f'regime_p{k}': post[i, k] for k in range(post.shape[1])}))
    return tab.merge(pd.DataFrame(rec), on=['cid', 't'], how='left'), exp_dur

## 8. RF vs LSTM on aligned 3-snapshot windows

In [ ]:
def window_arrays(tab, inst, feats, w=3):
    idx = tab.set_index(['cid', 't'])[feats]
    X, keep = [], []
    for i, r in inst.iterrows():
        keys = [(r.cid, r.t - k) for k in range(w - 1, -1, -1)]
        if all(k in idx.index for k in keys):
            block = idx.loc[keys].values
            if not np.isnan(block).any(): X.append(block); keep.append(i)
    return np.array(X), inst.loc[keep].reset_index(drop=True)

if HAS_TORCH:
    class SeqNet(nn.Module):
        def __init__(self, f, h=32):
            super().__init__(); self.lstm = nn.LSTM(f, h, batch_first=True); self.drop = nn.Dropout(0.3)
            self.head = nn.Linear(h, len(CLASSES))
        def forward(self, x):
            o, _ = self.lstm(x); return self.head(self.drop(o[:, -1]))

def train_lstm(Xtr, ytr, Xva, yva, seed, epochs=300, patience=25):
    torch.manual_seed(seed); np.random.seed(seed)
    net = SeqNet(Xtr.shape[2]); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    cnt = np.bincount(ytr, minlength=len(CLASSES)).astype(float)
    wts = torch.tensor(cnt.sum() / (len(CLASSES) * np.maximum(cnt, 1)), dtype=torch.float32)
    lossf = nn.CrossEntropyLoss(weight=wts)
    Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr)
    Xv = torch.tensor(Xva, dtype=torch.float32)
    best, best_state, wait = -1, None, 0
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xt))
        for b in range(0, len(Xt), 32):
            j = perm[b:b + 32]; opt.zero_grad(); lossf(net(Xt[j]), yt[j]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): pv = net(Xv).argmax(1).numpy()
        f = f1_score(yva, pv, average='macro', labels=range(len(CLASSES)), zero_division=0)
        if f > best: best, best_state, wait = f, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            wait += 1
            if wait > patience: break
    net.load_state_dict(best_state); net.eval()
    return net

def rf_vs_lstm(tab, inst, n_snap, rf_params, m=1, h=1):
    X, aligned = window_arrays(tab, inst, COMBINED)
    if len(aligned) < 40: return None
    cut = eval_cuts(n_snap)[0]
    tr = aligned.t.values + h + m <= cut; te = aligned.t.values >= cut     # purged, as in the main evaluation
    if te.sum() < 10 or tr.sum() < 30: return None
    y = aligned.label.map({c: i for i, c in enumerate(CLASSES)}).values
    sc = StandardScaler().fit(X[tr].reshape(-1, X.shape[2]))
    Xs = sc.transform(X.reshape(-1, X.shape[2])).reshape(X.shape)
    # validation = last 15% of training snapshots
    ttr = aligned.t.values[tr]; vcut = np.quantile(ttr, 0.85)
    trn, val = np.flatnonzero(tr & (aligned.t.values < vcut)), np.flatnonzero(tr & (aligned.t.values >= vcut))
    if len(val) < 5 or len(np.unique(y[trn])) < 2:
        trn, val = np.flatnonzero(tr), np.flatnonzero(tr)
    out = dict(n_train=int(tr.sum()), n_test=int(te.sum()))
    preds = {}
    rf_t = RandomForestClassifier(random_state=SEED, n_jobs=-1, **rf_params).fit(X[tr, -1], y[tr])
    preds['RF (snapshot t)'] = rf_t.predict(X[te, -1])
    rf_w = RandomForestClassifier(random_state=SEED, n_jobs=-1, **rf_params).fit(X[tr].reshape(tr.sum(), -1), y[tr])
    preds['RF (flattened t-2..t)'] = rf_w.predict(X[te].reshape(te.sum(), -1))
    if HAS_TORCH and RUN_LSTM:
        probs = []
        for s in range(CONFIG['LSTM_SEEDS']):
            net = train_lstm(Xs[trn], y[trn], Xs[val], y[val], seed=SEED + s)
            with torch.no_grad(): probs.append(torch.softmax(net(torch.tensor(Xs[te], dtype=torch.float32)), 1).numpy())
        preds['LSTM (t-2..t, seed ensemble)'] = np.mean(probs, 0).argmax(1)
    rows = []
    for k, p in preds.items():
        m = metrics(np.array(CLASSES)[y[te]], np.array(CLASSES)[p])
        rows.append(dict(model=k, **m))
    out['table'] = pd.DataFrame(rows)
    tests = []
    if 'LSTM (t-2..t, seed ensemble)' in preds:
        for k in ['RF (snapshot t)', 'RF (flattened t-2..t)']:
            a = preds[k] == y[te]; b = preds['LSTM (t-2..t, seed ensemble)'] == y[te]
            tbl = [[np.sum(a & b), np.sum(a & ~b)], [np.sum(~a & b), np.sum(~a & ~b)]]
            tests.append(dict(comparison=f'{k} vs LSTM', rf_only_correct=int(tbl[0][1]),
                              lstm_only_correct=int(tbl[1][0]), mcnemar_exact_p=mcnemar(tbl, exact=True).pvalue))
    out['mcnemar'] = pd.DataFrame(tests)
    return out

## 9. Survival analysis (community lifespan vs initial entropy)
Unit = community. Only communities **born after the first snapshot** are used (those present at $t=0$ are left-truncated). Two endpoints:
* **Primary — dissolution:** the community can no longer be tracked; communities still alive at the last snapshot are right-censored.
* **Secondary — first decline:** first *Shrinking* label (event at $t+h$) or dissolution, whichever comes first.

In the first full run the composite endpoint was degenerate (every community had the event, median survival = 1 snapshot in both groups), which is why dissolution is now primary. Kaplan–Meier by median initial entropy + log-rank, and Cox PH with z-scored initial entropy and log initial size.

In [ ]:
def survival_table(tab, n_snap, h=1):
    rows = []
    for cid, d in tab.groupby('cid'):
        d = d.sort_values('t'); t0, last = d.t.iloc[0], d.t.iloc[-1]
        if t0 == 0: continue
        dissolved = last < n_snap - 1
        t_diss = last + 1 if dissolved else last
        shr = d.loc[d.label == 'Shrinking', 't']
        t_shr = shr.iloc[0] + h if len(shr) else np.inf
        t_comp = min(t_shr, t_diss) if (len(shr) or dissolved) else last
        rows.append(dict(cid=cid, init_entropy=d.entropy.iloc[0], init_log_size=d.log_size.iloc[0],
                         dur_dissolution=t_diss - t0, ev_dissolution=int(dissolved),
                         dur_decline=t_comp - t0, ev_decline=int(len(shr) > 0 or dissolved)))
    return pd.DataFrame(rows)

def survival_analysis(st, endpoint='dissolution'):
    if st is None or st.empty: return None          # e.g. every community already present at t = 0
    st = st.rename(columns={f'dur_{endpoint}': 'duration', f'ev_{endpoint}': 'event'})
    st = st[st.duration > 0]
    if len(st) < 20 or st.event.sum() < 5 or st.event.sum() == len(st) and st.duration.nunique() < 2: return None
    med = st.init_entropy.median(); hi = st.init_entropy > med
    res = dict(n=len(st), events=int(st.event.sum()), censored=int((1 - st.event).sum()))
    kms = {}
    for name, mask in [('Low entropy', ~hi), ('High entropy', hi)]:
        km = KaplanMeierFitter().fit(st.duration[mask], st.event[mask], label=name); kms[name] = km
        res[f'median_survival_{name.split()[0].lower()}'] = km.median_survival_time_
    lr = logrank_test(st.duration[~hi], st.duration[hi], st.event[~hi], st.event[hi])
    res['logrank_p'] = lr.p_value
    cdf = st[['duration', 'event']].copy()
    cdf['z_init_entropy'] = zscore(st, ['init_entropy']).values.ravel()
    cdf['z_init_log_size'] = zscore(st, ['init_log_size']).values.ravel()
    cdf = cdf[[c for c in cdf.columns if c in ('duration', 'event') or cdf[c].std() > 0]]
    try:
        cph = CoxPHFitter(penalizer=0.01).fit(cdf, 'duration', 'event')
        s = cph.summary.loc['z_init_entropy']
        res.update(cox_HR_perSD=s['exp(coef)'], cox_HR_low=s['exp(coef) lower 95%'],
                   cox_HR_high=s['exp(coef) upper 95%'], cox_p=s['p'])
    except Exception as e:
        print('   Cox failed:', e)
    return res, kms

## 10. Per-dataset driver
Runs: load → snapshots → communities → triads → instances → prediction (3 feature sets + baselines) → ablation → SHAP → M1–M4 → survival → RF-vs-LSTM. Partitions and community statistics are cached so the robustness sweeps reuse them.

In [ ]:
CACHE, SNAPS, RESULTS = {}, {}, {}

EVENTS, WINDOW = {}, {}    # WINDOW[name] = (m, k): width = m * stride, label step k
def _mk(name, m=None, k=None):
    m0, k0 = WINDOW.get(name, (1, 1))
    return (m0 if m is None else m), (k0 if k is None else k)

def get_events(name):
    if name not in EVENTS:
        ev = load_events(name, DATASETS[name]); EVENTS[name] = ev
        print(f'  {name}: {len(ev):,} events, {pd.unique(ev[["src", "dst"]].values.ravel()).size:,} nodes, '
              f'{(ev.ts.max() - ev.ts.min()) / 86400:.0f} days')
    return EVENTS[name]

def get_snapshots(name, m=None):
    m, _ = _mk(name, m)
    if (name, m) not in SNAPS:
        s = DATASETS[name]['stride_days']
        SNAPS[(name, m)] = build_snapshots(get_events(name), m * s, s)
    return SNAPS[(name, m)]

def get_partition_stats(name, algo, seed, m=None):
    m, _ = _mk(name, m)
    key = (name, m, algo, seed)
    if key not in CACHE:
        snaps = get_snapshots(name, m); t = time.time()
        parts = [detect_communities(s['G'], algo, seed, CONFIG['MIN_COMM_SIZE']) for s in snaps]
        CACHE[key] = (parts, snapshot_stats(snaps, parts))
        print(f'  {name} [width {m}x{DATASETS[name]["stride_days"]}d, {algo}, seed={seed}]: '
              f'{len(snaps)} snapshots, {sum(map(len, parts))} community-snapshots ({time.time()-t:.0f}s)')
    return CACHE[key]

def make_table(name, algo=None, seed=None, theta=None, label_thr=None, m=None, k=None, death_as_shrink=None):
    algo = algo or CONFIG['CD_ALGO']; seed = CONFIG['CD_SEED'] if seed is None else seed
    theta = CONFIG['JACCARD_THETA'] if theta is None else theta
    label_thr = CONFIG['LABEL_THRESHOLD'] if label_thr is None else label_thr
    m, k = _mk(name, m, k)
    parts, stats = get_partition_stats(name, algo, seed, m)
    ids = track_communities(parts, theta)
    das = CONFIG['DEATH_AS_SHRINK'] if death_as_shrink is None else death_as_shrink
    return build_table(stats, ids, len(parts), label_thr, das, h=k)

In [ ]:
def mechanism_tests(inst, tab_reg):
    out, prim = {}, {}
    # M1 boundary closure -> growth
    d = inst.assign(y=(inst.label == 'Growing').astype(int))
    t1, f1 = cluster_fit(d, 'y', ['d_boundary_ratio', 'BCR', 'boundary_ratio', 'log_size', 'density', 'rel_d_size'])
    out['M1'] = (t1, f1)
    prim['M1'] = (t1.loc['d_boundary_ratio', 'p'], t1.loc['d_boundary_ratio', 'coef_perSD'], +1, t1.loc['d_boundary_ratio', 'se_cluster']) if t1 is not None else (np.nan, np.nan, +1, np.nan)
    # M2 internal closure -> trajectory
    grp = [inst.loc[inst.label == c, 'closure_rate'].values for c in CLASSES]
    m2 = dict(means={c: g.mean() for c, g in zip(CLASSES, grp)},
              ci95={c: stats.t.interval(0.95, len(g) - 1, g.mean(), stats.sem(g)) if len(g) > 1 else (np.nan, np.nan)
                    for c, g in zip(CLASSES, grp)})
    if all(len(g) > 1 for g in grp):
        m2['anova_F'], m2['anova_p'] = stats.f_oneway(*grp)
        m2['kruskal_H'], m2['kruskal_p'] = stats.kruskal(*grp)
        tk = pairwise_tukeyhsd(inst.closure_rate, inst.label)
        m2['tukey'] = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
        m2['cohens_d_G_vs_S'] = cohens_d(grp[0], grp[2])
        m2['cluster_boot_ci_G_minus_S'] = cluster_boot_diff(inst, 'closure_rate', 'Growing', 'Shrinking')
    dgs = inst[inst.label.isin(['Growing', 'Shrinking'])].assign(y=lambda x: (x.label == 'Growing').astype(int))
    t2, f2 = cluster_fit(dgs, 'y', ['closure_rate', 'log_size', 'density'])
    m2['logit'] = (t2, f2); out['M2'] = m2
    prim['M2'] = (t2.loc['closure_rate', 'p'], t2.loc['closure_rate', 'coef_perSD'], +1, t2.loc['closure_rate', 'se_cluster']) if t2 is not None else (np.nan, np.nan, +1, np.nan)
    # M3 entropy change -> shrinkage
    d = inst.assign(y=(inst.label == 'Shrinking').astype(int))
    t3, f3 = cluster_fit(d, 'y', ['entropy', 'd_entropy', 'log_size', 'rel_d_size'])
    t3b, f3b = cluster_fit(d, 'y', ['entropy', 'd_entropy'])   # unadjusted, draft-style specification
    out['M3'] = (t3, f3); out['M3_unadjusted'] = (t3b, f3b)
    prim['M3'] = (t3.loc['d_entropy', 'p'], t3.loc['d_entropy', 'coef_perSD'], +1, t3.loc['d_entropy', 'se_cluster']) if t3 is not None else (np.nan, np.nan, +1, np.nan)
    # M4 regime persistence -> lower volatility
    r = tab_reg.dropna(subset=['vol_next', 'regime_run', 'log_size']).assign(log_vol=lambda x: np.log1p(x.vol_next))
    # age control: run length grows mechanically with community age, and older communities are more stable
    t4, f4 = cluster_fit(r, 'log_vol', ['regime_run', 'regime_exp_dur', 'log_size', 'age'], kind='ols')
    t4b, f4b = cluster_fit(r, 'log_vol', ['regime_run', 'regime_exp_dur', 'log_size'], kind='ols')
    m4 = dict(ols=(t4, f4), ols_no_age=(t4b, f4b))
    if len(r) > 10:
        m4['spearman_run_vs_vol'] = stats.spearmanr(r.regime_run, r.vol_next)
        lab = r.dropna(subset=['label'])
        g = [lab.loc[lab.label == c, 'regime_run'] for c in CLASSES]
        if all(len(x) > 1 for x in g):
            m4['run_by_label_mean'] = {c: x.mean() for c, x in zip(CLASSES, g)}
            m4['run_by_label_kruskal_p'] = stats.kruskal(*g).pvalue
        gs = [lab.loc[lab.label == c, 'switch_rate'].dropna() for c in CLASSES]
        if all(len(x) > 1 for x in gs):
            m4['switch_by_label_mean'] = {c: x.mean() for c, x in zip(CLASSES, gs)}
            m4['switch_by_label_kruskal_p'] = stats.kruskal(*gs).pvalue
    out['M4'] = m4
    prim['M4'] = (t4.loc['regime_run', 'p'], t4.loc['regime_run', 'coef_perSD'], -1, t4.loc['regime_run', 'se_cluster']) if t4 is not None else (np.nan, np.nan, -1, np.nan)
    # Holm across the four primary tests
    keys = list(prim); ps = np.array([prim[k][0] for k in keys], float)
    holm = np.full(len(ps), np.nan); ok = ~np.isnan(ps)
    if ok.any(): holm[ok] = multipletests(ps[ok], method='holm')[1]
    names = {'M1': 'Boundary closure -> growth (d_boundary_ratio)', 'M2': 'Internal closure -> Growing vs Shrinking',
             'M3': 'Entropy change -> shrinkage (d_entropy)', 'M4': 'Regime persistence -> lower volatility (run length)'}
    summary = pd.DataFrame([dict(mechanism=names[k], coef_perSD=prim[k][1], se_perSD=prim[k][3], expected_sign='+' if prim[k][2] > 0 else '-',
                                 p_raw=prim[k][0], p_holm=h, verdict=verdict(h, prim[k][1], prim[k][2]))
                            for k, h in zip(keys, holm)])
    return out, summary

In [ ]:
def run_dataset(name):
    print(f'\n=== {name} ===')
    t_start = time.time(); R = dict(name=name, domain=DATASETS[name]['domain'])
    snaps = get_snapshots(name); n_snap = len(snaps); m, h = _mk(name)   # h = label step k
    R.update(m=m, k=h, width_days=m * DATASETS[name]['stride_days'], stride_days=DATASETS[name]['stride_days'],
             label_window_overlaps=h < m)
    tab = make_table(name); inst = instances_from(tab)
    R['n_snapshots'] = n_snap; R['n_instances'] = len(inst); R['n_communities'] = inst.cid.nunique()
    R['class_counts'] = inst.label.value_counts().reindex(CLASSES, fill_value=0).to_dict()
    print('  instances:', len(inst), R['class_counts'])
    if not WINDOW_OK.get(name, False):
        print('  !! no candidate window met the yield rule: reported in the descriptives, excluded from modelling.')
        R['skipped'] = True; return R
    FOLDS = make_folds(inst, n_snap, m, h)
    if not FOLDS:
        print('  !! no usable evaluation fold'); R['skipped'] = True; return R
    test = pd.concat([te for *_, te in FOLDS]); y, g = test.label.values, test.cid.values
    R['eval_cuts'] = [c for _, c, _, _ in FOLDS]; R['n_folds'] = len(FOLDS)
    R['n_train'] = len(FOLDS[0][2]); R['n_test'] = len(test); R['n_test_communities'] = test.cid.nunique()
    R['test_class_counts'] = test.label.value_counts().reindex(CLASSES, fill_value=0).to_dict()
    print(f"  folds at {R['eval_cuts']} | first-fold train {R['n_train']} | pooled test {R['n_test']} {R['test_class_counts']}")

    # ---- tuning (once per feature set, first fold's purged training data)
    seeds = [SEED + i for i in range(CONFIG['N_RF_SEEDS'])]
    params, cvf1 = {}, {}
    tr0 = FOLDS[0][2]
    for fs, cols in FEATURE_SETS.items():
        _, params[fs], cvf1[fs] = tune_rf(tr0[cols], tr0.label, tr0.cid)
    R['params_by_set'] = params; R['best_params'] = params['combined']

    # ---- rolling-origin predictions
    preds = {fs: [] for fs in FEATURE_SETS}; seed_preds = {fs: [] for fs in FEATURE_SETS}; fold_models = []
    base = {'majority': [], 'persistence': [], 'logreg': []}; strat = []
    for i, c, tr, te in FOLDS:
        for fs, cols in FEATURE_SETS.items():
            p, ps, est = rf_seed_ensemble(tr, te, cols, params[fs], seeds)
            preds[fs].append(p); seed_preds[fs].append(ps)
            if fs == 'combined': fold_models.append((est, tr, te))
        base['majority'].append(np.full(len(te), tr.label.mode()[0]))
        base['persistence'].append(persistence_baseline(te, CONFIG['LABEL_THRESHOLD']))
        lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')).fit(tr[COMBINED], tr.label)
        base['logreg'].append(lr.predict(te[COMBINED]))
        strat.append(np.tile(tr.label.value_counts(normalize=True).reindex(CLASSES).fillna(0).values, (len(te), 1)))
    P = {fs: np.concatenate(v) for fs, v in preds.items()}
    P.update({k_: np.concatenate(v) for k_, v in base.items()})

    perf = []
    for fs in FEATURE_SETS:
        mt = metrics(y, P[fs]); ci = cluster_bootstrap_ci(y, P[fs], g)
        per_seed = [macro_f1(y, np.concatenate([seed_preds[fs][f][s] for f in range(len(FOLDS))])) for s in range(len(seeds))]
        perf.append(dict(model=f'RF [{fs}]', cv_macro_f1=cvf1[fs], **mt, **{f'{k}_ci': ci[k] for k in ci},
                         macro_f1_single_seed_mean=np.mean(per_seed), macro_f1_single_seed_sd=np.std(per_seed)))
    for k_, lab in [('majority', 'Majority (per fold)'), ('persistence', 'Persistence (last change)'),
                    ('logreg', 'Logistic regression [combined]')]:
        mt = metrics(y, P[k_]); ci = cluster_bootstrap_ci(y, P[k_], g)
        perf.append(dict(model=lab, **mt, **{f'{kk}_ci': ci[kk] for kk in ci}))
    pv = np.cumsum(np.vstack(strat), 1); rng = np.random.default_rng(SEED)
    sims = [metrics(y, np.array(CLASSES)[(rng.random(len(y))[:, None] > pv).sum(1).clip(0, 2)]) for _ in range(200)]
    perf.append(dict(model='Stratified random (mean of 200 draws)', **{k: float(np.mean([s_[k] for s_ in sims])) for k in sims[0]}))
    R['performance'] = pd.DataFrame(perf)
    R['report'] = pd.DataFrame(classification_report(y, P['combined'], labels=CLASSES, output_dict=True, zero_division=0)).T
    R['confusion'] = pd.DataFrame(confusion_matrix(y, P['combined'], labels=CLASSES), index=CLASSES, columns=CLASSES)

    # ---- incremental value (paired cluster bootstrap on the same pooled instances)
    inc = []
    for a, b in [('combined', 'structural'), ('combined', 'size only'), ('macrostate', 'structural'),
                 ('macrostate', 'size only'), ('combined', 'persistence')]:
        d, ci, p = paired_cluster_boot(y, P[a], P[b], g)
        inc.append(dict(comparison=f'{a} - {b}', delta_macro_f1=d, ci_low=ci[0], ci_high=ci[1], p_boot=p))
    R['incremental'] = pd.DataFrame(inc)

    # ---- ablation (fixed tuned params, seed ensemble, paired bootstrap vs full model)
    abl = []
    for grp, drop in [('none (full model)', [])] + list(ABLATION_GROUPS.items()):
        cols = [c for c in COMBINED if c not in drop]
        pr = np.concatenate([rf_seed_ensemble(tr, te, cols, params['combined'],
                                              [SEED + 100 + s for s in range(CONFIG['N_ABLATION_SEEDS'])])[0]
                             for _, _, tr, te in FOLDS])
        if grp == 'none (full model)': full = pr
        d, ci, p = paired_cluster_boot(y, pr, full, g, n_boot=min(500, CONFIG['N_BOOT']))
        abl.append(dict(removed=grp, n_features=len(cols), macro_f1=macro_f1(y, pr), delta_vs_full=d,
                        delta_ci_low=ci[0], delta_ci_high=ci[1], p_boot=p))
    R['ablation'] = pd.DataFrame(abl)

    # ---- SHAP on each fold's test block (first-seed combined model) + fold-to-fold rank stability
    def shap_arr(model, X):
        sv = shap.TreeExplainer(model).shap_values(X)
        arr = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)
        return arr if arr.ndim == 3 else arr[..., None]
    parts_, ranks, pims = [], [], []
    for est, tr, te in fold_models:
        a_ = shap_arr(est, te[COMBINED]); cls = list(est.classes_)
        full_ = np.zeros((a_.shape[0], a_.shape[1], len(CLASSES)))
        for j, c in enumerate(cls): full_[:, :, CLASSES.index(c)] = a_[:, :, j]
        parts_.append(full_)
        ranks.append(pd.Series(np.abs(full_).mean((0, 2)), index=COMBINED).rank(ascending=False))
        if te.label.nunique() > 1:
            pim = permutation_importance(est, te[COMBINED], te.label, scoring='f1_macro', n_repeats=10, random_state=SEED)
            pims.append(pd.DataFrame({'mean': pim.importances_mean * len(te), 'n': len(te)}, index=COMBINED))
    arr = np.concatenate(parts_)
    sh = pd.DataFrame({f'mean|SHAP| {c}': np.abs(arr[:, :, i]).mean(0) for i, c in enumerate(CLASSES)}, index=COMBINED)
    sh['mean|SHAP| overall'] = sh.mean(1)
    rk = pd.concat(ranks, axis=1); sh['fold_rank_mean'] = rk.mean(1); sh['fold_rank_sd'] = rk.std(1)
    taus = [stats.kendalltau(rk.iloc[:, i], rk.iloc[:, j])[0] for i in range(rk.shape[1]) for j in range(i + 1, rk.shape[1])]
    R['shap_fold_kendall_tau_mean'] = float(np.mean(taus)) if taus else np.nan
    R['shap'] = sh.sort_values('mean|SHAP| overall', ascending=False)
    sh_feat = R['shap']['mean|SHAP| overall']
    R['shap_macro_share'] = float(sh_feat[MACRO].sum() / sh_feat.sum())   # share of attribution on macrostate features
    if pims:
        pm = sum(p_['mean'] for p_ in pims) / sum(p_['n'].iloc[0] for p_ in pims)
        R['perm_importance'] = pm.sort_values(ascending=False).to_frame('mean_weighted_by_fold_size')

    # ---- HMM (fit on the pre-evaluation period) + mechanisms
    hmm, bic, norm = fit_hmm(tab, R['eval_cuts'][0])
    tab_reg, exp_dur = add_regime_features(tab, hmm, norm)
    R['hmm_bic'] = bic; R['hmm_L'] = hmm.n_components
    R['hmm_L_at_grid_edge'] = hmm.n_components in (min(bic.L), max(bic.L))
    R['hmm_transmat'] = pd.DataFrame(hmm.transmat_); R['hmm_expected_duration'] = exp_dur
    R['hmm_means_shares'] = pd.DataFrame(hmm.means_ * norm[1] + norm[0], columns=HMM_COLS)
    R['mechanisms'], R['mechanism_summary'] = mechanism_tests(inst, tab_reg)

    # ---- survival
    st = survival_table(tab, n_snap, h)
    sv = survival_analysis(st, 'dissolution'); R['survival'], R['km'] = (sv if sv else (None, None))
    sv2 = survival_analysis(st, 'decline'); R['survival_decline'] = sv2[0] if sv2 else None

    # ---- RF vs LSTM (single purged split at the first cut)
    R['rf_vs_lstm'] = rf_vs_lstm(tab, inst, n_snap, params['combined'], m, h)
    R['_inst'], R['_tab'] = inst, tab_reg
    R['runtime_s'] = time.time() - t_start
    print(f'  done in {R["runtime_s"]:.0f}s')
    return R

## 10b. Window-width selection (outcome-blind)
For each dataset the stride is fixed (table at the top). The window width $m\cdot s$ and label step $k$ are chosen from `WINDOW_CANDIDATES`, tried **in the listed order**, by a **pre-specified rule that never looks at how features relate to outcomes**: the first candidate that yields ≥ `MIN_INSTANCES` labelled instances, ≥ `MIN_PER_CLASS` in every class, ≥ `MIN_COMMUNITIES` tracked communities, ≥ `MIN_TEST_INSTANCES` instances in the evaluation period (≥ `MIN_TEST_PER_CLASS` per class) and ≥ `MIN_TRAIN_INSTANCES` purged training instances before the first cut is used. All candidates use a non-overlapping label window (k = m), from the draft's design (disjoint windows, next-window label) to wider windows. The table also records how instances spread over the timeline (quartiles), which shows why a dataset fails.

If no candidate passes, the dataset is **excluded from modelling** and reported as lacking trackable community structure at this resolution. That is a substantive finding about where the method applies, not a failure to hide. The full yield table is saved (`table_window_selection.csv`) for the Methods section.

In [ ]:
def window_yield(name, m, k):
    parts, _ = get_partition_stats(name, CONFIG['CD_ALGO'], CONFIG['CD_SEED'], m)
    ids = track_communities(parts, CONFIG['JACCARD_THETA'])
    chain = Counter(c for ii in ids for c in ii)
    inst = instances_from(make_table(name, m=m, k=k))
    cc = inst.label.value_counts().reindex(CLASSES, fill_value=0)
    return dict(dataset=name, m=m, k=k, width_days=m * DATASETS[name]['stride_days'], stride_days=DATASETS[name]['stride_days'],
                label_window_overlaps=k < m,
                snapshots=len(parts), community_snapshots=sum(map(len, parts)),
                tracked_ids=len(chain), ids_seen_3plus=sum(v >= 3 for v in chain.values()),
                median_chain=float(np.median(list(chain.values()))) if chain else 0,
                instances=len(inst), communities=inst.cid.nunique() if len(inst) else 0,
                **cc.to_dict(), min_class=int(cc.min()), **_eval_yield(inst, len(parts), m, k))

def _eval_yield(inst, n_snap, m, k):
    c0 = eval_cuts(n_snap)[0]
    if len(inst) == 0:
        return dict(first_cut=c0, train_first_fold=0, test_instances=0, test_min_class=0, instances_by_time_quartile='0/0/0/0')
    te, tr = inst[inst.t >= c0], purged_train(inst, c0, m, k)
    q = np.histogram(inst.t, bins=np.linspace(0, n_snap, 5))[0]
    return dict(first_cut=c0, train_first_fold=len(tr), test_instances=len(te),
                test_min_class=int(te.label.value_counts().reindex(CLASSES, fill_value=0).min()),
                instances_by_time_quartile='/'.join(map(str, q)))

def passes(y):
    return (y['instances'] >= CONFIG['MIN_INSTANCES'] and y['min_class'] >= CONFIG['MIN_PER_CLASS']
            and y['communities'] >= CONFIG['MIN_COMMUNITIES'] and y['test_instances'] >= CONFIG['MIN_TEST_INSTANCES']
            and y['test_min_class'] >= CONFIG['MIN_TEST_PER_CLASS'] and y['train_first_fold'] >= CONFIG['MIN_TRAIN_INSTANCES'])

SELECTION, WINDOW_OK = [], {}
for name in DATASETS:
    print(f'\n--- {name}')
    try:
        tried = []
        for m, k in CONFIG['WINDOW_CANDIDATES']:
            y = window_yield(name, m, k); tried.append(y)
            print(f"   width {y['width_days']:>4}d, label step {k}{' (overlapping)' if k < m else ''}: {y['instances']:>5} instances, "
                  f"classes {[y[c] for c in CLASSES]}, {y['communities']} communities, median chain {y['median_chain']:.0f} | "
                  f"eval {y['test_instances']} (min class {y['test_min_class']}), train {y['train_first_fold']}, "
                  f"by time quartile {y['instances_by_time_quartile']}")
            if passes(y): break
        ok_ = [y for y in tried if passes(y)]
        best = ok_[0] if ok_ else max(tried, key=lambda y: (y['min_class'], y['instances']))
        WINDOW[name] = (best['m'], best['k']); WINDOW_OK[name] = bool(ok_)
        for y in tried: y.update(chosen=(y is best), meets_rule=passes(y))
        SELECTION += tried
        print(f"   -> {'chosen' if ok_ else 'NO candidate met the rule; best was'}: width {best['width_days']}d, label step {best['k']}")
    except Exception as e:
        import traceback; traceback.print_exc(); print(f'!! {name}: selection failed: {e}')
SELECTION = pd.DataFrame(SELECTION)
SELECTION.to_csv(os.path.join(RESULTS_DIR, 'table_window_selection.csv'), index=False)
display(SELECTION)

## 11. Run all datasets

In [ ]:
for name in DATASETS:
    try:
        RESULTS[name] = run_dataset(name)
    except Exception as e:
        import traceback; traceback.print_exc(); print(f'!! {name} failed: {e}')
    with open(os.path.join(RESULTS_DIR, 'results_checkpoint.pkl'), 'wb') as f:
        pickle.dump({k: {kk: vv for kk, vv in v.items() if kk not in ('km',)} for k, v in RESULTS.items()}, f)
OK = [n for n, r in RESULTS.items() if not r.get('skipped')]
print('Completed:', OK)
if not OK:
    raise RuntimeError('No dataset met the yield rule, so there is nothing to model. See table_window_selection.csv; '
                       'consider adding a stride/width candidate or a dataset with persistent membership.')

## 12. Dataset descriptives

In [ ]:
desc = pd.DataFrame([dict(dataset=n, domain=r['domain'], stride_days=DATASETS[n]['stride_days'],
                          width_days=WINDOW.get(n, (1, 1))[0] * DATASETS[n]['stride_days'], label_step=WINDOW.get(n, (1, 1))[1],
                          meets_yield_rule=WINDOW_OK.get(n, False),
                          snapshots=r['n_snapshots'], communities=r.get('n_communities'),
                          instances=r['n_instances'], **r['class_counts'],
                          train_first_fold=r.get('n_train'), eval_pooled=r.get('n_test'), eval_cuts=r.get('eval_cuts'))
                     for n, r in RESULTS.items()])
desc.to_csv(os.path.join(RESULTS_DIR, 'table_datasets.csv'), index=False); desc

## 13. Predictive performance (purged rolling origin, pooled; 95 % cluster-bootstrap CI) and incremental value

In [ ]:
rows = []
for n in OK:
    p = RESULTS[n]['performance'].copy(); p.insert(0, 'dataset', n); rows.append(p)
perf_all = pd.concat(rows, ignore_index=True)
perf_all.to_csv(os.path.join(RESULTS_DIR, 'table_performance.csv'), index=False)
show = perf_all.copy()
for k in ['accuracy', 'macro_f1', 'balanced_acc']:
    cis = show[f'{k}_ci'] if f'{k}_ci' in show else [None] * len(show)
    show[k] = [fmt_ci(v, c) if isinstance(c, tuple) else f'{v:.3f}' for v, c in zip(show[k], cis)]
display(show[['dataset', 'model', 'cv_macro_f1', 'accuracy', 'macro_f1', 'balanced_acc', 'macro_f1_single_seed_sd']])

inc_all = pd.concat([RESULTS[n]['incremental'].assign(dataset=n) for n in OK], ignore_index=True)
inc_all.to_csv(os.path.join(RESULTS_DIR, 'table_incremental.csv'), index=False)
print('Incremental macro-F1 (paired cluster bootstrap):'); display(inc_all.round(4))

# random-effects meta-analysis of the incremental value across datasets (SE from the bootstrap CI)
meta_inc = []
for comp, d in inc_all.groupby('comparison'):
    res = meta_random_effects(d.delta_macro_f1, (d.ci_high - d.ci_low) / (2 * 1.96))
    if res: meta_inc.append(dict(comparison=comp, **res))
meta_inc = pd.DataFrame(meta_inc); meta_inc.to_csv(os.path.join(RESULTS_DIR, 'table_meta_incremental.csv'), index=False)
print('Pooled across datasets (random effects, HKSJ):'); display(meta_inc.round(4))
for n in OK:
    print(f'\n{n}: per-class report (combined RF, pooled folds)'); display(RESULTS[n]['report'].round(3))

## 14. Ablation

In [ ]:
abl_all = pd.concat([RESULTS[n]['ablation'].assign(dataset=n) for n in OK], ignore_index=True)
abl_all.to_csv(os.path.join(RESULTS_DIR, 'table_ablation.csv'), index=False)
print('delta macro-F1 vs full model (negative = the removed group was helping):')
display(abl_all.pivot(index='removed', columns='dataset', values='delta_vs_full').round(3))
abl_all['delta [95% CI]'] = [f'{d:+.3f} [{lo:+.3f}, {hi:+.3f}]' for d, lo, hi in zip(abl_all.delta_vs_full, abl_all.delta_ci_low, abl_all.delta_ci_high)]
display(abl_all.pivot(index='removed', columns='dataset', values='delta [95% CI]'))
meta_abl = []
for grp, d in abl_all[abl_all.removed != 'none (full model)'].groupby('removed'):
    res = meta_random_effects(d.delta_vs_full, (d.delta_ci_high - d.delta_ci_low) / (2 * 1.96))
    if res: meta_abl.append(dict(removed=grp, **res))
meta_abl = pd.DataFrame(meta_abl); meta_abl.to_csv(os.path.join(RESULTS_DIR, 'table_meta_ablation.csv'), index=False)
print('Pooled ablation deltas (random effects, HKSJ):'); display(meta_abl.round(4))

## 15. SHAP attribution (quantitative) and rank stability

In [ ]:
for n in OK:
    sh = RESULTS[n]['shap']; sh.to_csv(os.path.join(RESULTS_DIR, f'table_shap_{n}.csv'))
    if 'perm_importance' in RESULTS[n]: RESULTS[n]['perm_importance'].to_csv(os.path.join(RESULTS_DIR, f'table_permimp_{n}.csv'))
    print(f'{n}: Kendall tau of SHAP rankings between evaluation folds = {RESULTS[n]["shap_fold_kendall_tau_mean"]:.2f}; '
          f'share of mean|SHAP| on macrostate features = {RESULTS[n]["shap_macro_share"]:.2f}')
    display(sh.head(10).round(4))

## 16. Mechanism tests M1–M4 (Holm within dataset, BH across datasets)

In [ ]:
mech = pd.concat([RESULTS[n]['mechanism_summary'].assign(dataset=n) for n in OK], ignore_index=True)
ok = mech.p_raw.notna()
mech['p_BH_all'] = np.nan
mech.loc[ok, 'p_BH_all'] = multipletests(mech.loc[ok, 'p_raw'], method='fdr_bh')[1]
mech.to_csv(os.path.join(RESULTS_DIR, 'table_mechanisms_summary.csv'), index=False)
display(mech[['dataset', 'mechanism', 'coef_perSD', 'se_perSD', 'expected_sign', 'p_raw', 'p_holm', 'p_BH_all', 'verdict']].round(4))
print('\nVerdict counts per mechanism across datasets:')
display(mech.groupby(['mechanism', 'verdict']).size().unstack(fill_value=0))

# pooled (random-effects) estimate per mechanism, Holm across the four pooled tests
meta_m = []
for mech_name, d in mech.groupby('mechanism', sort=False):
    res = meta_random_effects(d.coef_perSD, d.se_perSD)
    if res: meta_m.append(dict(mechanism=mech_name, expected_sign=d.expected_sign.iloc[0],
                               n_same_sign_as_expected=int((np.sign(d.coef_perSD) == (1 if d.expected_sign.iloc[0] == '+' else -1)).sum()), **res))
meta_m = pd.DataFrame(meta_m)
if len(meta_m):
    meta_m['p_holm'] = multipletests(meta_m.p, method='holm')[1]
    meta_m['verdict'] = [verdict(ph, mu, 1 if es == '+' else -1) for ph, mu, es in zip(meta_m.p_holm, meta_m.pooled, meta_m.expected_sign)]
meta_m.to_csv(os.path.join(RESULTS_DIR, 'table_meta_mechanisms.csv'), index=False)
print('Pooled mechanism effects (per-SD coefficient; random effects, HKSJ; Holm across the 4 mechanisms):')
display(meta_m.round(4))

In [ ]:
# Detailed tables per mechanism
detail_rows = []
for n in OK:
    M = RESULTS[n]['mechanisms']
    for key in ['M1', 'M3', 'M3_unadjusted']:
        t, f = M[key]
        if t is not None:
            detail_rows.append(t.assign(dataset=n, model=key, **{k: v for k, v in f.items()}).reset_index(names='term'))
    t, f = M['M2']['logit']
    if t is not None: detail_rows.append(t.assign(dataset=n, model='M2_logit', **f).reset_index(names='term'))
    t, f = M['M4']['ols']
    if t is not None: detail_rows.append(t.assign(dataset=n, model='M4_ols', **f).reset_index(names='term'))
    t, f = M['M4']['ols_no_age']
    if t is not None: detail_rows.append(t.assign(dataset=n, model='M4_ols_no_age', **f).reset_index(names='term'))
detail = pd.concat(detail_rows, ignore_index=True)
detail.to_csv(os.path.join(RESULTS_DIR, 'table_mechanism_models.csv'), index=False)

m2 = []
for n in OK:
    d = RESULTS[n]['mechanisms']['M2']
    m2.append(dict(dataset=n, **{f'mean_{c}': d['means'][c] for c in CLASSES},
                   anova_F=d.get('anova_F'), anova_p=d.get('anova_p'), kruskal_H=d.get('kruskal_H'),
                   kruskal_p=d.get('kruskal_p'), cohens_d_G_vs_S=d.get('cohens_d_G_vs_S'),
                   boot_ci_G_minus_S=tuple(np.round(d['cluster_boot_ci_G_minus_S'], 4)) if 'cluster_boot_ci_G_minus_S' in d else None))
    if 'tukey' in d: d['tukey'].to_csv(os.path.join(RESULTS_DIR, f'table_tukey_closure_{n}.csv'), index=False)
m2 = pd.DataFrame(m2); m2.to_csv(os.path.join(RESULTS_DIR, 'table_M2_closure.csv'), index=False)
display(m2.round(4))
display(detail[detail.model.isin(['M1', 'M3', 'M4_ols', 'M4_ols_no_age'])][['dataset', 'model', 'term', 'coef_perSD', 'p', 'OR_perSD', 'OR_low', 'OR_high']].round(4))

## 17. Latent regimes (HMM)

In [ ]:
hm = []
for n in OK:
    r = RESULTS[n]; m4 = r['mechanisms']['M4']
    hm.append(dict(dataset=n, L_by_BIC=r['hmm_L'], L_at_grid_edge=r['hmm_L_at_grid_edge'],
                   expected_durations=np.round(r['hmm_expected_duration'], 2).tolist(),
                   spearman_run_vs_vol=m4.get('spearman_run_vs_vol', (np.nan, np.nan))[0],
                   spearman_p=m4.get('spearman_run_vs_vol', (np.nan, np.nan))[1],
                   run_by_label=m4.get('run_by_label_mean'), run_by_label_kruskal_p=m4.get('run_by_label_kruskal_p'),
                   switch_by_label=m4.get('switch_by_label_mean'), switch_kruskal_p=m4.get('switch_by_label_kruskal_p')))
    r['hmm_bic'].to_csv(os.path.join(RESULTS_DIR, f'table_hmm_bic_{n}.csv'), index=False)
hm = pd.DataFrame(hm); hm.to_csv(os.path.join(RESULTS_DIR, 'table_hmm.csv'), index=False); display(hm)

## 18. Survival

In [ ]:
sv = pd.DataFrame([dict(dataset=n, endpoint='dissolution', **RESULTS[n]['survival']) for n in OK if RESULTS[n].get('survival')])
sv2 = pd.DataFrame([dict(dataset=n, endpoint='first decline', **RESULTS[n]['survival_decline']) for n in OK if RESULTS[n].get('survival_decline')])
pd.concat([sv, sv2]).to_csv(os.path.join(RESULTS_DIR, 'table_survival.csv'), index=False)
display(sv.round(4)); display(sv2.round(4))
if len(sv) == 0: print('No dataset had enough post-birth communities/events for survival analysis.')
else:
  fig, axes = plt.subplots(1, max(1, len(sv)), figsize=(3.2 * max(1, len(sv)), 2.8), squeeze=False)
  for ax, n in zip(axes[0], sv.dataset):
      for (lbl, km), col in zip(RESULTS[n]['km'].items(), ['#0072B2', '#E69F00']):
          km.plot_survival_function(ax=ax, ci_show=True, color=col)
      ax.set_title(f'{n}\nlog-rank p={RESULTS[n]["survival"]["logrank_p"]:.2g}'); ax.set_xlabel('snapshots since birth')
      ax.set_ylabel('P(still tracked)')
  plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'fig_km.png'), dpi=300); plt.show()

## 19. RF vs LSTM

In [ ]:
rl = []
for n in OK:
    o = RESULTS[n].get('rf_vs_lstm')
    if not o: continue
    rl.append(o['table'].assign(dataset=n, n_train=o['n_train'], n_test=o['n_test']))
    if len(o['mcnemar']): display(o['mcnemar'].assign(dataset=n))
if rl:
    rl = pd.concat(rl, ignore_index=True); rl.to_csv(os.path.join(RESULTS_DIR, 'table_rf_vs_lstm.csv'), index=False); display(rl.round(3))

## 20. Cross-domain transfer matrix
Train on the purged pre-evaluation period of dataset A (combined features, A's tuned hyper-parameters, seed ensemble) and evaluate on the evaluation period of dataset B. The diagonal equals within-domain performance. Two variants: raw features, and features **z-scored within each dataset** using that dataset's own training-period statistics (unsupervised, no target labels used), which separates scale shift from a genuine change in the feature→outcome mapping.

In [ ]:
def transfer_matrix(norm):
    M = pd.DataFrame(index=OK, columns=OK, dtype=float)
    data = {}
    for n in OK:
        inst = RESULTS[n]['_inst']; tr, te, _ = temporal_split(inst, RESULTS[n]['n_snapshots'], RESULTS[n]['m'], RESULTS[n]['k'])
        if norm:
            mu, sd = tr[COMBINED].mean(), tr[COMBINED].std(ddof=0).replace(0, 1)
            tr = tr.assign(**((tr[COMBINED] - mu) / sd)); te = te.assign(**((te[COMBINED] - mu) / sd))
        data[n] = (tr, te)
    seeds = [SEED + i for i in range(CONFIG['N_RF_SEEDS'])]
    for a in OK:
        for b in OK:
            te = data[b][1]
            M.loc[a, b] = macro_f1(te.label, rf_seed_ensemble(data[a][0], te, COMBINED, RESULTS[a]['best_params'], seeds)[0])
    return M

if len(OK) < 2:
    print('Cross-domain transfer needs at least two completed datasets; skipped.'); TRANSFER = {}
else:
  TRANSFER = {'raw': transfer_matrix(False), 'z-scored per dataset': transfer_matrix(True)}
  fig, axes = plt.subplots(1, 2, figsize=(4.2 * 2 + 1, 3.8))
  for ax, (k, M) in zip(axes, TRANSFER.items()):
      M.to_csv(os.path.join(RESULTS_DIR, f'table_transfer_{k.split()[0]}.csv'))
      im = ax.imshow(M.values.astype(float), cmap='Blues', vmin=0, vmax=1)
      ax.set_xticks(range(len(OK))); ax.set_xticklabels(OK, rotation=45, ha='right')
      ax.set_yticks(range(len(OK))); ax.set_yticklabels(OK); ax.grid(False)
      ax.set_xlabel('test dataset'); ax.set_ylabel('train dataset'); ax.set_title(f'Macro-F1 transfer ({k})')
      for i in range(len(OK)):
          for j in range(len(OK)):
              v = M.values[i, j]; ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='white' if v > 0.6 else 'black')
  fig.colorbar(im, ax=axes, shrink=0.8)
  plt.savefig(os.path.join(RESULTS_DIR, 'fig_transfer.png'), dpi=300, bbox_inches='tight'); plt.show()
for k, M in TRANSFER.items():
    diag = np.diag(M.values.astype(float)); off = M.values.astype(float)[~np.eye(len(OK), dtype=bool)]
    print(f'{k}: mean within-domain macro-F1 = {diag.mean():.3f}; mean cross-domain = {off.mean():.3f}')

## 21. Robustness sweeps
For every modelled dataset (or the list in `ROBUSTNESS_DATASETS`), the core outcomes are recomputed under the same purged rolling-origin protocol while varying one choice at a time: window width and label step (including overlapping labels), Jaccard tracking threshold θ, the ±label threshold, the Louvain seed, the detection algorithm (Leiden, label propagation), and whether dissolution counts as shrinkage. RF hyper-parameters are fixed at the main run's tuned values (re-tuning per setting would conflate robustness of the finding with robustness of the search).

In [ ]:
def core_outcomes(name, **kw):
    m, h = _mk(name, kw.get('m'), kw.get('k'))
    tab = make_table(name, **kw); inst = instances_from(tab); n_snap = len(get_snapshots(name, m))
    o = dict(N=len(inst), communities=inst.cid.nunique(), **{c: int((inst.label == c).sum()) for c in CLASSES})
    if len(inst) < 60 or inst.label.nunique() < 3: return o
    F = make_folds(inst, n_snap, m, h)
    if not F or sum(len(te) for *_, te in F) < 20: return o
    pbs = RESULTS[name]['params_by_set']; y = np.concatenate([te.label.values for *_, te in F])
    g = np.concatenate([te.cid.values for *_, te in F])
    pr = {fs: np.concatenate([RandomForestClassifier(random_state=SEED, n_jobs=-1, **pbs[fs])
                              .fit(tr[FEATURE_SETS[fs]], tr.label).predict(te[FEATURE_SETS[fs]]) for _, _, tr, te in F])
          for fs in ['combined', 'structural']}
    o['N_eval'] = len(y); o['macro_f1'] = macro_f1(y, pr['combined'])
    o['f1_combined_minus_structural'] = o['macro_f1'] - macro_f1(y, pr['structural'])
    gr = [inst.loc[inst.label == c, 'closure_rate'] for c in CLASSES]
    o['closure_d_G_vs_S'] = cohens_d(gr[0], gr[2])
    for key, d, xs, term in [
            ('M1', inst.assign(y=(inst.label == 'Growing').astype(int)), ['d_boundary_ratio', 'BCR', 'boundary_ratio', 'log_size', 'density', 'rel_d_size'], 'd_boundary_ratio'),
            ('M2', inst[inst.label.isin(['Growing', 'Shrinking'])].assign(y=lambda x: (x.label == 'Growing').astype(int)), ['closure_rate', 'log_size', 'density'], 'closure_rate'),
            ('M3', inst.assign(y=(inst.label == 'Shrinking').astype(int)), ['entropy', 'd_entropy', 'log_size', 'rel_d_size'], 'd_entropy')]:
        t, _ = cluster_fit(d, 'y', xs)
        if t is not None: o[f'{key}_{term}_coef'], o[f'{key}_{term}_p'] = t.loc[term, 'coef_perSD'], t.loc[term, 'p']
    return o

ROB = []
if RUN_ROBUSTNESS:
    rd = CONFIG['ROBUSTNESS_DATASETS']
    rob_sets = OK if rd == 'all' else ([d for d in rd if d in OK] or OK[:2])
    for n in rob_sets:
        t0_ = time.time(); print(f'robustness: {n}')
        for mm, kk in CONFIG['ROBUSTNESS_WINDOWS']:
            ROB.append(dict(dataset=n, sweep='window (width d / label step)', value=f"{mm * DATASETS[n]['stride_days']}/{kk}",
                            overlapping=kk < mm, **core_outcomes(n, m=mm, k=kk)))
        for th in CONFIG['JACCARD_SWEEP']:   ROB.append(dict(dataset=n, sweep='jaccard_theta', value=th, **core_outcomes(n, theta=th)))
        for lt in CONFIG['LABEL_SWEEP']:     ROB.append(dict(dataset=n, sweep='label_threshold', value=lt, **core_outcomes(n, label_thr=lt)))
        for sd in CONFIG['CD_SEED_SWEEP']:   ROB.append(dict(dataset=n, sweep='louvain_seed', value=sd, **core_outcomes(n, seed=sd)))
        for al in CONFIG['CD_ALGO_SWEEP']:
            if al == 'leiden' and not HAS_LEIDEN: continue
            ROB.append(dict(dataset=n, sweep='cd_algorithm', value=al, **core_outcomes(n, algo=al)))
        for das in [False, True]:            ROB.append(dict(dataset=n, sweep='death_as_shrink', value=das, **core_outcomes(n, death_as_shrink=das)))
        print(f'  {time.time() - t0_:.0f}s')
        pd.DataFrame(ROB).to_csv(os.path.join(RESULTS_DIR, 'table_robustness.csv'), index=False)   # checkpoint
    ROB = pd.DataFrame(ROB); ROB.to_csv(os.path.join(RESULTS_DIR, 'table_robustness.csv'), index=False)
    display(ROB.round(4))
    # sign stability: share of settings in which each effect keeps the main-run sign
    stab = []
    for n, d in ROB.groupby('dataset'):
        for col in ['f1_combined_minus_structural', 'closure_d_G_vs_S', 'M1_d_boundary_ratio_coef', 'M2_closure_rate_coef', 'M3_d_entropy_coef']:
            if col in d and d[col].notna().any():
                v = d[col].dropna(); stab.append(dict(dataset=n, quantity=col, settings=len(v), share_positive=(v > 0).mean(),
                                                      median=v.median(), min=v.min(), max=v.max()))
    stab = pd.DataFrame(stab); stab.to_csv(os.path.join(RESULTS_DIR, 'table_robustness_sign_stability.csv'), index=False)
    display(stab.round(3))

In [ ]:
if RUN_ROBUSTNESS and len(ROB):
    num = ROB[ROB.sweep.isin(['jaccard_theta', 'label_threshold'])].copy(); num['value'] = num.value.astype(float)
    metrics_to_plot = [('macro_f1', 'Macro-F1 (pooled folds)'), ('f1_combined_minus_structural', 'Macro-F1: combined − structural'),
                       ('closure_d_G_vs_S', "Cohen's d closure (G vs S)"),
                       ('M3_d_entropy_coef', 'ΔH coef. (shrink logit, per SD)'), ('M1_d_boundary_ratio_coef', 'Δboundary coef. (growth logit)')]
    fig, axes = plt.subplots(2, len(metrics_to_plot), figsize=(3.1 * len(metrics_to_plot), 5.2), squeeze=False)
    pal = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#D55E00', '#000000', '#999999', '#F0E442']
    for r, sweep in enumerate(['jaccard_theta', 'label_threshold']):
        for c, (col, title) in enumerate(metrics_to_plot):
            ax = axes[r, c]
            for k, (n, d) in enumerate(num[num.sweep == sweep].groupby('dataset')):
                if col in d: ax.plot(d.value, d[col], marker='o', ms=4, lw=2, color=pal[k % len(pal)], label=n)
            ax.axhline(0, color='#999', lw=0.8) if 'coef' in col or 'd_G' in col else None
            ax.set_title(title, fontsize=8); ax.set_xlabel(sweep.replace('_', ' '))
    axes[0, 0].legend(fontsize=7, frameon=False)
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'fig_robustness.png'), dpi=300); plt.show()

## 22. Per-dataset figures

In [ ]:
for n in OK:
    r = RESULTS[n]; inst = r['_inst']
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.1))
    cc = pd.Series(r['class_counts']).reindex(CLASSES)
    axes[0].bar(CLASSES, cc.values, color=[CLASS_COLORS[c] for c in CLASSES], width=0.6)
    axes[0].set_title('Labelled instances'); [axes[0].text(i, v, str(v), ha='center', va='bottom', fontsize=8) for i, v in enumerate(cc.values)]
    data = [inst.loc[inst.label == c, 'closure_rate'] for c in CLASSES]
    bp = axes[1].boxplot(data, showfliers=False, patch_artist=True, widths=0.5)
    axes[1].set_xticks([1, 2, 3]); axes[1].set_xticklabels(CLASSES)
    for patch, c in zip(bp['boxes'], CLASSES): patch.set_facecolor(CLASS_COLORS[c]); patch.set_alpha(0.55)
    p = r['mechanisms']['M2'].get('anova_p', np.nan); axes[1].set_title(f'Internal closure rate (ANOVA p={p:.2g})')
    sh = r['shap']['mean|SHAP| overall'].head(10)[::-1]
    axes[2].barh(sh.index, sh.values, color='#0072B2', height=0.6); axes[2].set_title('Top-10 mean |SHAP|')
    cm = r['confusion'].values; axes[3].imshow(cm, cmap='Blues'); axes[3].grid(False)
    axes[3].set_xticks(range(3)); axes[3].set_xticklabels(CLASSES); axes[3].set_yticks(range(3)); axes[3].set_yticklabels(CLASSES)
    for i in range(3):
        for j in range(3): axes[3].text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    axes[3].set_xlabel('predicted'); axes[3].set_ylabel('true'); axes[3].set_title('Confusion (test)')
    fig.suptitle(f'{n} ({r["domain"]})', fontsize=10); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'fig_{n}.png'), dpi=300); plt.show()

## 23. Manuscript summary
One line per dataset with the numbers that the abstract, results and discussion should quote. Every value here is traceable to a CSV in `RESULTS_DIR`.

In [ ]:
summ = []
for n in OK:
    r = RESULTS[n]; p = r['performance'].set_index('model'); m2 = r['mechanisms']['M2']
    comb = p.loc['RF [combined]']; ms = r['mechanism_summary'].set_index('mechanism')['verdict']
    inc = r['incremental'].set_index('comparison')
    summ.append({'dataset': n, 'window width/stride (d), label step': f"{r['width_days']}/{r['stride_days']}, k={r['k']}"
                 + (' (overlapping)' if r['label_window_overlaps'] else ''),
                 'N instances (G/St/Sh)': f"{r['n_instances']} ({'/'.join(str(r['class_counts'][c]) for c in CLASSES)})",
                 'N evaluated (pooled folds)': r['n_test'],
                 'macro-F1 [95% CI]': fmt_ci(comb.macro_f1, comb.macro_f1_ci),
                 'seed sd': round(comb.macro_f1_single_seed_sd, 3),
                 'stratified-random macro-F1': round(p.loc['Stratified random (mean of 200 draws)', 'macro_f1'], 3),
                 'majority acc': round(p.filter(like='Majority', axis=0).accuracy.iloc[0], 3),
                 'persistence macro-F1': round(p.loc['Persistence (last change)', 'macro_f1'], 3),
                 'size-only macro-F1': round(p.loc['RF [size only]', 'macro_f1'], 3),
                 'struct-only macro-F1': round(p.loc['RF [structural]', 'macro_f1'], 3),
                 'Δ comb.−struct. [CI]': fmt_ci(inc.loc['combined - structural', 'delta_macro_f1'],
                                                (inc.loc['combined - structural', 'ci_low'], inc.loc['combined - structural', 'ci_high'])),
                 'Δ comb.−size [CI]': fmt_ci(inc.loc['combined - size only', 'delta_macro_f1'],
                                             (inc.loc['combined - size only', 'ci_low'], inc.loc['combined - size only', 'ci_high'])),
                 'macro-only macro-F1': round(p.loc['RF [macrostate]', 'macro_f1'], 3),
                 "closure d (G vs S)": round(m2.get('cohens_d_G_vs_S', np.nan), 3),
                 'M1': ms.iloc[0], 'M2': ms.iloc[1], 'M3': ms.iloc[2], 'M4': ms.iloc[3],
                 'HMM L (BIC)': f"{r['hmm_L']}{' (grid edge)' if r['hmm_L_at_grid_edge'] else ''}",
                 'SHAP share macrostate': round(r['shap_macro_share'], 2),
                 'Cox HR entropy, dissolution [CI]': (f"{r['survival']['cox_HR_perSD']:.2f} [{r['survival']['cox_HR_low']:.2f}, {r['survival']['cox_HR_high']:.2f}]"
                                         if r.get('survival') and 'cox_HR_perSD' in r['survival'] else 'n/a')})
summ = pd.DataFrame(summ); summ.to_csv(os.path.join(RESULTS_DIR, 'table_manuscript_summary.csv'), index=False)
display(summ)

meta = dict(pipeline_version=PIPELINE_VERSION, windows={n: dict(m=m, k=k, meets_rule=WINDOW_OK.get(n)) for n, (m, k) in WINDOW.items()},
            config=CONFIG, datasets={k: {kk: vv for kk, vv in v.items()} for k, v in DATASETS.items()},
            versions=dict(python=sys.version, networkx=nx.__version__, shap=shap.__version__,
                          torch=torch.__version__ if HAS_TORCH else None),
            run_finished=time.strftime('%Y-%m-%d %H:%M:%S'))
with open(os.path.join(RESULTS_DIR, 'run_metadata.json'), 'w') as f: json.dump(meta, f, indent=2, default=str)

In [ ]:
# Package everything for download
archive = shutil.make_archive(os.path.join(BASE_DIR, 'results_macrostates'), 'zip', RESULTS_DIR)
print('Wrote', archive)
if IN_COLAB:
    if SAVE_TO_DRIVE:
        from google.colab import drive; drive.mount('/content/drive')
        shutil.copy(archive, '/content/drive/MyDrive/'); print('Copied to Drive')
    from google.colab import files; files.download(archive)

## 24. Notes for the manuscript
* **One protocol.** Quote pooled rolling-origin numbers from `table_performance.csv` with their cluster-bootstrap CIs. CV scores are for tuning only.
* **The central claim is incremental value**, from `table_incremental.csv` (per dataset, paired cluster bootstrap) and `table_meta_incremental.csv` (pooled across datasets). Beating the majority class is not the claim: beating size-only, structural-only and persistence is.
* **Windows.** Report stride, chosen width and label step per dataset from `table_window_selection.csv` (including the evaluation-period yield and the instance distribution over time), and the window sweep, including overlapping labels, from `table_robustness.csv`. Datasets that fail the yield rule are listed in `table_datasets.csv` (`meets_yield_rule = False`): the method needs communities whose membership persists over several windows.
* **Mechanisms.** Per-dataset verdicts come from `table_mechanisms_summary.csv` (Holm within dataset, BH across all tests). The headline is the pooled estimate in `table_meta_mechanisms.csv` (random effects, HKSJ, Holm across the four mechanisms), reported together with I² and the number of datasets agreeing in sign.
* **Robustness.** `table_robustness_sign_stability.csv` gives, per dataset, the share of alternative settings in which each effect keeps its sign. Report it next to every claimed effect.
* **Disclose the v2 → v3 changes** marked † in the analysis plan at the top: the protocol was revised after the v2 run, before v3 was run.
* **M4 and community age.** Run length grows with community age, so the primary M4 model controls for age. `M4_ols_no_age` is the uncontrolled version.
* **M3 specification.** Entropy of triad shares is mechanically linked to size change, so the primary model controls for log size and the previous size change. `M3_unadjusted` reproduces the draft's two-predictor specification.
* **Triad definition.** Only *connected* triples (≥ 2 edges) with ≥ 2 community members are counted. State this explicitly in Methods.
* **Survival.** Primary endpoint = dissolution (no longer trackable). Secondary = first decline. Only communities born after the first snapshot are used. Report median lifetimes: when most communities live 1–2 snapshots, the survival analysis is descriptive at best.
* **Datasets.** SNAP's Slashdot and Enron e-mail networks are static (no timestamps), so they cannot support snapshot-based community tracking. Only timestamped SNAP networks are used.